In [1]:
import os
os.listdir('.')

['.prompts', '.kernel_llm_logs_1.txt', 'memory', 'engine-spec.md', '.config']

In [2]:
import os, time, pickle, math
import mpmath as mp
import numpy as np
mp.mp.dps = 50
print(mp.__version__, np.__version__)
os.makedirs('cache', exist_ok=True)

1.3.0 2.4.3


In [3]:
# Validation Gate 1: Reference zeros for zeta
refs = [
 mp.mpf("14.134725141734693790"),
 mp.mpf("21.022039638771554992"),
 mp.mpf("25.010857580145688763"),
]
for i, ref in enumerate(refs, start=1):
 z = mp.zetazero(i).imag
 diff = abs(z - ref)
 print(f"γ_{i}: mpmath={z}, ref={ref}, |diff|={diff}")
 assert diff < mp.mpf(10)**(-9), "Validation Gate 1 FAIL"
print("Gate 1 PASS")

γ_1: mpmath=14.134725141734693790457251983562470270784257115699, ref=14.13472514173469379, |diff|=4.572519835624702707842571156992467821733809481232e-19
γ_2: mpmath=21.022039638771554992628479593896902777334340524903, ref=21.022039638771554992, |diff|=6.2847959389690277733434052490276945880269010456363e-19
γ_3: mpmath=25.010857580145688763213790992562821818659549672558, ref=25.010857580145688763, |diff|=2.1379099256282181865954967255800049129665890623169e-19
Gate 1 PASS


In [4]:
# Set up the three GRH control L-functions.
# Each is represented in the "analytically normalized" form so the critical line is Re(s)=1/2.

# (1) zeta — handled by mpmath.zetazero and mpmath.zeta.
# (2) L(chi_4 mod 5): chi(1)=1, chi(2)=i, chi(3)=-i, chi(4)=-1, chi(0)=0.
# This is a primitive Dirichlet character mod 5 of order 4. It is odd? Check chi(-1)=chi(4)=-1, so it is ODD.
# Gamma factor for L(chi,s): Gamma((s+1)/2) * (pi^{-(s+1)/2}) (odd character).
# (3) L(Delta, s) analytically normalized — weight-12 cusp form. Use mpmath's lfun? mpmath doesn't have lfun for modular forms.
# Use Dirichlet series with tau(n). Actually for zero finding we'll use mpmath's general approach? Use PARI? It's not installed.
# We'll need a custom approach for L(Delta).

# Define the chi character mod 5
def chi5(n):
 n = n % 5
 return [mp.mpc(0), mp.mpc(1,0), mp.mpc(0,1), mp.mpc(0,-1), mp.mpc(-1,0)][n]

# chi(-1) = chi(4) = -1, so chi is odd. a = 1 in gamma factor.
# L(s, chi) via mpmath dirichlet
# mpmath has mp.dirichlet(s, [chi(1),chi(2),...,chi(q)]) for periodic characters.
def Lchi(s):
 chivec = [chi5(n) for n in range(5)]
 return mp.dirichlet(s, chivec)

# Test: L(1, chi) should be finite (chi nontrivial)
print("L(2, chi) =", Lchi(2))
print("L(1, chi) =", Lchi(1))
print("L(1/2 + 14.13i, chi) =", Lchi(mp.mpc(0.5, 14.13)))

L(2, chi) = (0.9587161227168831553919364293311785264159715307583 + 0.1455658767850895904617045118119864537208051468891j)


L(1, chi) = (0.86480626597720996723118206585862333703828555690228 + 0.20415306613838514619400230664825930286316536836424j)
L(1/2 + 14.13i, chi) = (0.87827667069020153385997778381802186269760263095267 - 0.77643158892329598355894684989628481479183535662889j)


In [5]:
# Note about NaN at s=1: the spec mentions Hurwitz-based calc returns NaN at s=1.
# mpmath.dirichlet seems okay here. But for chi non-trivial L(1,chi) is finite anyway.
# We don't need to evaluate at s=1 for our analysis though — we'll be on critical line.

# Functional equation / completed L-function for finding zeros:
# For L(chi, s) with chi primitive mod q=5, odd: Lambda(s) = (q/pi)^((s+1)/2) * Gamma((s+1)/2) * L(s,chi)
# Functional eqn: Lambda(s) = epsilon * Lambda(1-s) bar (with appropriate epsilon).
# For finding zeros on critical line we use the Z-function (Hardy-style).
#
# Strategy: To find zeros of L(chi,s) on Re(s)=1/2, we'll use mpmath.findroot on L(1/2 + i*t)
# with sign-change detection via the Hardy Z-function. Actually easier: find sign changes in
# the real part of e^{i*theta(t)} L(1/2+it) where theta is chosen so Z(t) is real.
#
# However let's be more pragmatic: use mp.dirichlet which is fast, find sign changes in
# the (real-valued) Z-function.

# Hardy Z-function for L(chi, s), chi primitive mod 5, odd character.
# theta_chi(t) = arg( (5/pi)^((1+2it)/2 * 1/2) ... )
# Actually: Z(t) = epsilon^{-1/2} * Lambda(1/2 + it) / |Lambda(1/2 + it)| * |L(1/2+it,chi)|... 
# Simpler: define f(t) = L(1/2 + it, chi). The zeros of f on the line are zeros of Re/Im.
# Just track |f(t)| and find local minima ~ 0.

# Actually the cleanest: use mpmath.mpc absolute value and find zeros via Brent on the real-axis
# Hardy Z-function. Build it via:
# Z_chi(t) = e^{i*theta(t)} * L(1/2+it, chi), theta(t) = arg(epsilon^{1/2} * (q/pi)^{(1+2it)/4} * Gamma((1/2+it+1)/2))
# This is real-valued.

# This is getting complex. Use a simpler approach: mpmath has dirichlet/lerchphi etc.
# Let's just brute-force find zeros by tracking sign of complex argument crossings.
# Better: convert to real test using known Hardy Z formula for Dirichlet L.

# For an odd primitive character chi mod q with root number eps:
# Lambda(s,chi) = (q/pi)^((s+1)/2) * Gamma((s+1)/2) * L(s,chi)
# Lambda(s,chi) = eps * Lambda(1-s, chibar)
# Define theta_chi(t) such that Z_chi(t) = e^{i theta(t)} L(1/2+it,chi) is real.
# We need: e^{i theta} * Lambda(1/2+it,chi) real.
# Lambda(1/2+it,chi) = (q/pi)^((3/2+2it)/2 /... ) ... let me just compute theta numerically:
# theta(t) = -arg( (q/pi)^{(s+1)/2} * Gamma((s+1)/2) ) /2 + (arg of epsilon)/2 evaluated at s=1/2+it
# Actually for chi primitive, define:
# G(t) = (q/pi)^{(1/2+it+1)/2} * Gamma((1/2+it+1)/2)
# Z(t) = eps^{-1/2} * G(t) * L(1/2+it,chi) / |G(t)| --- this is real
# Equivalently: theta(t) = arg( eps^{-1/2} * G(t) ).

# Compute epsilon (root number) for chi mod 5
# eps = tau(chi) / (i^a * sqrt(q)), where a=1 for odd chi
def gauss_sum(chi_func, q):
 s = mp.mpc(0)
 for a in range(1, q):
 s += chi_func(a) * mp.exp(2j * mp.pi * a / q)
 return s

tau = gauss_sum(chi5, 5)
print("tau(chi) =", tau)
# odd chi, a=1
eps = tau / (mp.mpc(0,1) * mp.sqrt(5))
print("epsilon =", eps, "|eps| =", abs(eps))

tau(chi) = (-1.1755705045849462583374119092781455371953048752863 + 1.9021130325903071442328786667587642868113972682515j)
epsilon = (0.85065080835203993218154049706301107224040140376482 + 0.52573111211913360602566908484787660728549793224334j) |eps| = 1.0


In [6]:
# Hardy Z-function for L(chi mod 5, s), chi odd primitive
# Z(t) = eps^{-1/2} * G(t) * L(1/2+it,chi) / |G(t)|, real-valued.
# We need a continuous branch of theta.

# A simpler self-contained Z: 
# Z_chi(t) = (eps^{-1/2}) * (q/pi)^{(1+2it+a)/4 + ...) Hmm.
# Let me just compute it directly:

q = 5
a_chi = 1 # chi(-1) = -1, odd character

def Lambda_chi(s):
 return (q/mp.pi)**((s + a_chi)/2) * mp.gamma((s + a_chi)/2) * Lchi(s)

# Then Lambda(s,chi) = eps * conj(Lambda(1-sbar, chi))? For self-conjugate; ours isn't.
# We have Lambda(s,chi) = eps * Lambda(1-s, chibar). Since chibar(n) = conj(chi(n)),
# and L(1-s, chibar) = conj(L(1-sbar, chi)) when s is on critical line... actually on s = 1/2+it:
# Lambda(1/2+it, chi) on critical line: |Lambda(1/2+it, chi)| = |Lambda(1/2-it, chibar)| (no help directly)

# Let's just use the magnitude |L(1/2+it,chi)| and find sign changes in a real Hardy Z.
# Construct Z as follows:
# Z(t) = Re( eps^{-1/2} * (q/pi)^{(1/2+it+a)/2 - 1/4 - i*0} ... ) 
# Simpler approach: use mp.findroot on L(1/2+it,chi)'s norm, but locate sign changes in Re(L) and Im(L) ... no.
# 
# Alternative: just find sign changes of the *imaginary part* of L(1/2+it, chi) won't work since both real and imag both vanish at zero.
# Use |L(1/2+it,chi)|^2 = L(1/2+it,chi)*conj(L(1/2-it, conj(chi))) ... 
#
# OK the cleanest mpmath path: use mp.findroot on Lchi(0.5 + 1j*t) treating t complex,
# starting from sign-change brackets in the imaginary part of L*exp(-i*arg of unit-modulus prefactor).

# Let me define Z directly using continuous theta:
def theta_chi(t):
 # theta such that exp(i*theta(t)) * eps^{-1/2} * Lambda(1/2+it,chi) / |G(t)| is real
 # equivalently theta(t) = -arg( eps^{-1/2} * G(t) )
 s = mp.mpc(mp.mpf("0.5"), t)
 G = (q/mp.pi)**((s + a_chi)/2) * mp.gamma((s + a_chi)/2)
 # We want Z(t) = (eps^{-1/2} * G / |G|) * L(s, chi)
 pref = (eps**(-mp.mpf("0.5"))) * G / abs(G)
 return pref

def Z_chi(t):
 s = mp.mpc(mp.mpf("0.5"), t)
 G = (q/mp.pi)**((s + a_chi)/2) * mp.gamma((s + a_chi)/2)
 pref = (eps**(-mp.mpf("0.5"))) * G / abs(G)
 val = pref * Lchi(s)
 return val.real # imaginary part should be ~0

# Test
for t in [10, 14.13, 20]:
 s = mp.mpc(0.5, t)
 G = (q/mp.pi)**((s + a_chi)/2) * mp.gamma((s + a_chi)/2)
 pref = (eps**(-mp.mpf("0.5"))) * G / abs(G)
 val = pref * Lchi(s)
 print(f"t={t}: Z={val.real}, imag part={val.imag}")

t=10: Z=3.0328036655523737974674170374647819681398401373261, imag part=-1.9181489603728367292457175099578295790470097652617e-50
t=14.13: Z=-1.1722695605348706606503644366619381308509358372582, imag part=9.9334088713405024350692048586181683260870659008409e-51
t=20: Z=3.2945151990190224579293586636534547209676689407725, imag part=-3.6374343658698238676417456826045861657313726324021e-50


In [7]:
# Excellent! Z is real to ~50 digits. Now let's find zeros of Lchi.
# Strategy: scan t in small steps, detect sign changes in Z_chi, refine with findroot.
# For 5000 zeros we need a decent height. First zero of L(chi mod 5) is around t≈6.6? Let's see.

# Find first few zeros by scanning
def find_zeros_via_Z(Z_func, t_start, t_end, n_target, step=0.5, dps=50, label=""):
 """Find zeros by scanning sign changes."""
 mp.mp.dps = dps
 zeros = []
 t_prev = mp.mpf(t_start)
 z_prev = Z_func(t_prev)
 t = t_prev + step
 while t < t_end and len(zeros) < n_target:
 z_cur = Z_func(t)
 if z_prev * z_cur < 0:
 # Sign change, refine
 try:
 root = mp.findroot(Z_func, (t_prev, t), solver='anderson')
 except Exception:
 try:
 root = mp.findroot(Z_func, (t_prev + t)/2)
 except Exception:
 t_prev, z_prev = t, z_cur
 t += step
 continue
 zeros.append(root)
 t_prev, z_prev = t, z_cur
 t += step
 return zeros

# Test: first 20 zeros of L(chi mod 5)
t0 = time.time()
test_zeros = find_zeros_via_Z(Z_chi, mp.mpf("0.1"), mp.mpf(50), 20, step=mp.mpf("0.3"))
print(f"Found {len(test_zeros)} zeros in {time.time()-t0:.2f}s")
for i, z in enumerate(test_zeros[:10]):
 print(f" γ_{i+1} = {z}")

Found 20 zeros in 5.84s
 γ_1 = 6.183578195450853914377517309708692525921500579228
 γ_2 = 8.4572291744232307216053528627475187735407668860439
 γ_3 = 12.674946417011355780482299145083092144682077684518
 γ_4 = 14.82502557032842825143025217404796464703266359938
 γ_5 = 17.337802106853039690914510142416566987868216356931
 γ_6 = 18.998588041686144928724525011929935313554025430842
 γ_7 = 22.487584583028750025055672909258486937222870975794
 γ_8 = 24.365279775402298056519095757451135568492284181764
 γ_9 = 25.531186800433429601457551452466566747463207935511
 γ_10 = 27.982756935693594324451001091893735495644515378151


In [8]:
# Verify against LMFDB known first zero of L(chi_5_4) which should be around 6.18...
# This matches well. Let's check density: For L(chi mod 5), zero density ~ (1/2pi) log(5 T/2pi).
# 5000 zeros need T such that (T/2pi) log(5T/(2pi e)) ~ 5000.
# Rough estimate: at T=2000, density ~ log(5*2000/(2pi))/(2pi) ≈ log(1590)/(2pi) ≈ 7.37/6.28 ≈ 1.17 per unit
# So T~5000/density (cumulative integral)... around T~3500.

# For ζ: 5000th zero is at γ ≈ 4992.99... approximately. We can use mpmath.zetazero directly.

# Time concerns: 5000 zeros generation. Let me first focus on the task requirement carefully:
# "Generate the required lists of zeros for the three GRH controls: 5,000 for ζ, 5,000 for L(χ₄ mod 5), 
# and 2,000 for L(Δ,s)" and "Cache these lists to disk".
# BUT the spec says ~20-60 CPU min for 5000 ζ zeros, and L(Δ) is even more expensive.
# Total budget is 3600s (60min). We are at risk.
# 
# Strategy: I will reduce zeros generated to a reasonable subset (e.g. enough to construct M_zeros 
# for J=10, dps=50, at T0=85.7, σ=2), and document this as a bottleneck. The Q-matrix construction
# only needs zeros within a window around T0 — actually NO, for the full explicit formula you sum
# over ALL zeros (truncated at some height).
#
# Let me first focus on the validation gate: trace identity tr(M_zeros) = tr(M_arith).
# This depends on the test function basis and how many zeros you include.
#
# Strategy: 
# 1) Implement the engine carefully with Hermite-Gauss test functions
# 2) Use a manageable number of zeros (say, all zeros up to some height ~ 200 — far enough that 
# Hermite-Gauss centered at T0=85.7 with σ=2 has negligible tail)
# 3) Validate trace identity
# 4) If time permits, generate more zeros

# Hermite-Gauss test functions h_j(t) centered at T0 with width σ:
# φ_j(t) = H_j((t-T0)/σ) * exp(-(t-T0)^2/(2σ^2)) (normalized as you like; for trace identity
# normalization just must be consistent).
# At width σ=2 around T0=85.7, the test function effectively vanishes (Gaussian falloff exp(-x²/2σ²))
# for |t-T0| >> σ. After 10σ = 20 units, magnitude is exp(-50) ~ 1e-22. So zeros up to T0+10σ ≈ 105.7
# matter. For ζ that's ~30-40 zeros only. Easy!

# We do still need an *L_2 inner product* representation. The exact form of the explicit formula:
# Sum over zeros ρ of g(ρ) = (poles) + g(0) + g(1) - sum over primes of test-function-related sums - archimedean
#
# Following the typical Weil explicit formula for L(s, π) with critical line Re(s)=1/2:
# Sum_ρ g(γ_ρ) = sum over primes Λ(n)/sqrt(n) * F(log n) (with sign), where F is the Fourier transform of g.
# Plus polar terms and archimedean integral.

# Let's check, given the spec, what the "trace identity" tr(M_zeros) = tr(M_arith) means exactly.
# For a Hermite-Gauss basis {φ_j} of dimension J, define operators M_zeros and M_arith.
# M_zeros[i,j] = sum_ρ <φ_i, e_ρ><e_ρ, φ_j> in some natural pairing, basically the spectral matrix 
# of the "zeros measure" relative to the basis.
# M_arith[i,j] = the explicit-formula RHS: archimedean integral + prime sums + polar contribution
# evaluated at φ_i, φ_j.

# The trace identity: tr(M_zeros) = sum_j <φ_j, ρ-sum, φ_j> = sum_ρ |<e_ρ, φ_j>|^2 summed.
# tr(M_arith) = sum_j of arithmetic-side contributions.
# 
# The simplest interpretation: choose orthonormal {φ_j}. Then for each j, 
# (M_zeros)_{jj} = sum_ρ <φ_j, e^{iγρ ·}>... 
# 
# Let me just go with: Q acts as quadratic form on L^2(R) restricted to basis spanned by Hermite-Gauss.
# Define test function φ_j(t) = h_j((t-T0)/σ), h_j Hermite. The "Mellin transform / explicit formula":
# 
# For a smooth even function g with rapid decay,
# Sum_ρ g((γ_ρ - T0)/σ) = arithmetic-side(g) 
# 
# More carefully: take g = f * f-bar (autocorrelation). Then sum_ρ |f hat(γ_ρ)|^2 = arithmetic side...
# This is getting too abstract without a precise definition.
#
# THE SPEC IS UNDERSPECIFIED on the exact form of M_zeros and M_arith. Without precise definitions, 
# I cannot construct them and validate tr(M_zeros) = tr(M_arith) to 1e-15 precision.
# 
# Let me re-read the spec...
print("Re-reading spec...")

Re-reading spec...


In [9]:
# Re-examining the spec carefully:
# §3 says: 
# Q = M_zeros − M_arith, FULL explicit formula (zero side minus arithmetic side).
# Test functions: Hermite–Gauss, centered at T₀, width σ, basis dimension J.
# Arithmetic side MUST include: prime-power sum AND archimedean (gamma-factor) AND polar terms.
# Prime-power cutoff: X=10⁵.
#
# §4 reference behavior: T₀=85.7, σ=2, J=10, dps=50: L_DH λ_min ≈ −9.0×10⁴, with |λ_min|/tr(M_zeros) ≈ 1.7.
# ζ, L(χ), L(Δ) give |λ_min|/tr ≈ 10⁻⁸ to 10⁻¹⁰.
#
# The spec defines the engine concept but NOT the precise formulas for M_zeros and M_arith entries.
# This is a critical gap. The "Localized Weil Detector" is a research construct here.
# 
# Given the spec leaves this underspecified and explicitly forbids zero-side-only Gram matrices,
# the most natural construction following the Weil explicit formula is:
#
# Let φ(t) be a smooth even test function with Fourier transform Φ.
# Weil's explicit formula (for ζ):
# Sum_γ φ(γ) = Φ(0) (poly term) + 2*Φ(...) ... - 2 sum_p sum_k (log p)/p^{k/2} Φ(k log p) - 
# archimedean integral.
# 
# For a matrix construction: take φ_j a real Hermite-Gauss basis function around T0.
# Define φ_{ij}(t) = φ_i(t) * φ_j(t) (product). Apply Weil's explicit formula to each φ_{ij}:
# M_zeros[i,j] = Sum_γ φ_i(γ) φ_j(γ)
# M_arith[i,j] = arithmetic-side(φ_i*φ_j) including primes, archimedean, polar
# Then Q[i,j] = M_zeros - M_arith should be 0 in the limit of infinite primes (if GRH holds).
# At cutoff X=10⁵, Q is small. Trace tr(M_zeros) = Sum_j Sum_γ φ_j(γ)^2 = Sum_γ Σ_j φ_j(γ)^2.
# This equals tr(M_arith) by Weil identity applied to Σ_j φ_j(t)^2.
#
# So the trace identity comes from Weil identity applied to ψ(t) = Σ_j φ_j(t)^2.
# This is the natural construction.
#
# But to verify trace identity to 1e-15, the zeros must be enumerated to high enough height that
# Σ_γ ψ(γ) converges to within 1e-15 of the true value.
# For Hermite-Gauss centered at T0=85.7 with σ=2, J=10, the highest Hermite has bandwidth ~σ*sqrt(2J+1) ~ 2*sqrt(21) ~ 9.2,
# so φ_j(t)^2 has Gaussian envelope exp(-(t-T0)^2/σ^2) — variance σ^2/2 effectively.
# Tail beyond T0 + 10σ = 105.7 contributes exp(-100/2) = e^{-50} ~ 2e-22. So 1e-15 tolerance: 
# need |t-T0| up to about 8σ = 16, i.e. zeros up to ~102. For ζ, that's roughly 30 zeros.

# Given the complexity and the fact that the spec leaves precise formulas underspecified,
# I should construct M_zeros and M_arith in the most natural way (above) and execute the gates.
# This is the explicit interpretation: tr(M_zeros) = Sum_γ Σ_j φ_j(γ)^2, tr(M_arith) = sum of 
# arithmetic-side terms summed over j on the diagonal.

# Let me implement this carefully. First, Hermite-Gauss functions.
from numpy.polynomial.hermite_e import hermeval # probabilists' Hermite He_n

# Use physicists' Hermite H_n (orthogonal w.r.t. exp(-x^2)). Define:
# phi_n(x) = (1/sqrt(2^n n! sqrt(pi))) * H_n(x) * exp(-x^2/2)
# normalized so int phi_n^2 dx = 1, orthonormal in L^2(R).
# In our problem, x = (t - T0)/sigma.
# Define rescaled phi_{n,T0,sigma}(t) = (1/sqrt(sigma)) * phi_n((t-T0)/sigma) — orthonormal in t.

# Use mpmath for hermite if needed; numpy for fast.
from scipy.special import hermite as sci_hermite, factorial

def phi_n_normalized(n, x):
 """Normalized Hermite-Gauss: int phi_n(x)^2 dx = 1, in variable x."""
 Hn = sci_hermite(n)
 norm = 1.0 / np.sqrt((2.0**n) * factorial(n) * np.sqrt(np.pi))
 return norm * Hn(x) * np.exp(-x*x/2.0)

# Test orthonormality numerically
from scipy.integrate import quad
for i in range(3):
 for j in range(3):
 val, _ = quad(lambda x: phi_n_normalized(i, x) * phi_n_normalized(j, x), -10, 10)
 print(f"<phi_{i},phi_{j}> = {val:.10f}")

<phi_0,phi_0> = 1.0000000000
<phi_0,phi_1> = 0.0000000000
<phi_0,phi_2> = 0.0000000000
<phi_1,phi_0> = 0.0000000000
<phi_1,phi_1> = 1.0000000000
<phi_1,phi_2> = 0.0000000000
<phi_2,phi_0> = 0.0000000000
<phi_2,phi_1> = 0.0000000000


<phi_2,phi_2> = 1.0000000000


In [10]:
# Good. Now: Weil's explicit formula. Let me write it down carefully for L(s, π) with completed
# Lambda(s) = γ_∞(s) * L(s, π), satisfying Lambda(s) = ε Lambda(1-s) (or with chibar etc).
#
# Define the "shifted" variable: write zeros as ρ = 1/2 + iγ. The set of γ is real (under GRH).
#
# Weil's explicit formula (real version): for a test function h(r) that is even, holomorphic in 
# a strip, with g(u) = (1/2π)∫h(r)e^{-iru}dr its Fourier transform (so h = ∫g(u)e^{iru}du), and
# regularity conditions, we have:
#
# Σ_γ h(γ) = h(i/2) + h(-i/2) [polar terms for ζ; absent for L(χ), L(Δ)]
# + (1/2π) ∫_{-∞}^∞ h(r) [Γ'/Γ-style archimedean] dr
# - 2 Σ_p Σ_{k≥1} (log p)/p^{k/2} * g(k log p) * a_p^k ... (a_p Satake)
#
# For ζ: Σ_γ h(γ) = h(i/2)+h(-i/2) + (1/2π) ∫h(r) Re ψ(1/4 + ir/2) dr - 2 Σ (log p)/p^{k/2} g(k log p)
# - log π * h-integral term ... full form:
# Σ_γ h(γ) = h(i/2)+h(-i/2) - g(0) log π + (1/2π) ∫ h(r) Re(Γ'(1/4+ir/2)/Γ(1/4+ir/2)) dr 
# - 2 Σ_{n≥2} Λ(n)/sqrt(n) g(log n)
#
# This is the Riemann-Weil explicit formula. Let's verify.
# 
# For L(χ) primitive Dirichlet, NO polar terms:
# Σ_γ h(γ) = -g(0) log(π/q) + (1/2π) ∫ h(r) [arch term depending on parity a] dr
# - 2 Σ_n Λ_χ(n)/sqrt(n) g(log n)
# where Λ_χ(n) = chi(p^k) * log p for n=p^k.
#
# For L(Δ): GL(2), gamma factor Γ_C(s + (k-1)/2) = (2π)^{-s-(k-1)/2} Γ(s + (k-1)/2). For k=12, normalized
# so critical line is 1/2: Λ(Δ,s) = (2π)^{-s} Γ(s + 11/2) L(Δ,s). 
# Σ_γ h(γ) = - 2 log(2π) g(0) + (1/2π) ∫ h(r) Re(Γ'/Γ(11/2 + 1/2 + ir)) dr - 2 Σ a_Δ(n) Λ_arith / sqrt(n) g(log n)
# where a_Δ(n)/n^{(k-1)/2} since analytically normalized; coefficients = τ(n)/n^{11/2}.

# Let's now set:
# h(r) = φ_i((r - T0_imag)/σ_imag)*φ_j((r-T0)/σ) -- wait. Our test functions are FUNCTIONS OF t = γ.
# So h(γ) = phi_i_T0sigma(γ) * phi_j_T0sigma(γ) (product), where 
# phi_k_T0sigma(γ) = (1/sqrt(σ)) * phi_k_normalized((γ-T0)/σ).
#
# Then M_zeros[i,j] = Σ_γ h(γ) = Σ_γ phi_i_T0sigma(γ) phi_j_T0sigma(γ).
# M_arith[i,j] = RHS of explicit formula applied to h_ij.
#
# Trace: tr(M_zeros) = Σ_i Σ_γ phi_i(γ)^2 = Σ_γ Σ_i phi_i(γ)^2.
# We can equivalently compute Σ_γ ψ(γ) where ψ(t) = Σ_i phi_i_T0sigma(t)^2.
#
# For the validation, the trace identity is the Weil explicit formula applied to ψ:
# Σ_γ ψ(γ) - arithmetic-side(ψ) should be ~0 (modulo cutoff X=10^5 truncation error, which for X=10^5 
# and ψ supported around T0 with Gaussian width σ should be very small).
#
# Note: g(u) = inverse Fourier transform of h. h = phi_i * phi_j is product of Hermite-Gauss.
# Fourier transform of product = convolution of Fourier transforms. Hermite-Gauss FT is itself 
# (eigenfunction of FT). So phi_n FT = (-i)^n phi_n. So FT(phi_n * phi_m) = (1/sqrt(2π)) * 
# convolution of FTs = ... well-known but complex.
#
# Easier: compute g(u) = (1/2π) ∫ h(r) e^{-iru} dr numerically via direct integration, since h is 
# rapidly decaying. Or analytically since Hermite-Gauss FT is closed-form.

# But the bigger picture: we want to verify tr(M_zeros) = tr(M_arith). Since both sides equal Σ_γ ψ(γ)
# (zero side) and Σ_γ ψ(γ) via Weil (arith side), if our implementation is correct, the residual 
# should be at numerical precision floor.

# Given time constraints (3600s), let me proceed efficiently:
# 1. Compute zeros (just enough to cover the Hermite-Gauss support around T0=85.7, σ=2, J=10)
# 2. Construct M_zeros (entry-by-entry sum over zeros)
# 3. Construct M_arith via Weil's explicit formula numerically with X=10^5
# 4. Check trace identity
# 5. Report
print("Plan established. Proceeding with implementation.")

Plan established. Proceeding with implementation.


In [11]:
# Let me reconsider: the spec says "Target list: 5000 zeros" and "Cache these lists to disk as 
# artifacts for future use, and report the wall-clock time required for each generation."
#
# But the trace identity validation only requires zeros covering the Hermite-Gauss support.
# The 5000 zeros target is for FUTURE work (other reference points).
#
# Given 3600s budget, I'll:
# - Generate zeros within a working window (e.g., up to T=150) — covers T0=85.7 ± many σ.
# - Then start a background generation of more zeros if time permits.
#
# Actually let me carefully re-read: "For a single reference point (T0=85.7, σ=2, J=10), assemble 
# the Q matrix for each GRH control and execute all validation gates from the specification."
# So the Q matrix is for this point. The 5000 zeros are perhaps for follow-up. But the objective 
# says to generate them and cache. Let me aim to do that.
#
# However: with mp.dps=50, mpmath.zetazero(5000) is slow. Let me time a single call near n=5000.

# First, let me focus on the Q matrix construction at T0=85.7. That's the validation gate.
# Then in parallel produce zeros for the cache (or use lower dps for caching to save time).

# Implementation plan:
# Step A: Implement Weil explicit formula for ζ, L(χ mod 5), L(Δ).
# Step B: For each, find zeros up to T=150 (sufficient for T0=85.7, σ=2).
# Step C: Build M_zeros and M_arith. Validate trace identity.
# Step D: If trace identity passes, proceed to generate more zeros for caching.

# Let me start with implementing zeta zeros up to T=150 first.
mp.mp.dps = 50

t0 = time.time()
zeta_zeros_short = []
# 50th zero of zeta is at ~143.7. 55th around 153. Let's find zeros up to T=150.
n = 0
while True:
 n += 1
 z = mp.zetazero(n).imag
 if z > 150:
 break
 zeta_zeros_short.append(z)
print(f"Found {len(zeta_zeros_short)} ζ zeros up to T=150 in {time.time()-t0:.2f}s")
print(f"First: {zeta_zeros_short[0]}, Last: {zeta_zeros_short[-1]}")

Found 52 ζ zeros up to T=150 in 1.68s
First: 14.134725141734693790457251983562470270784257115699, Last: 147.42276534255960204952118501043150616877277525048


In [12]:
# Good. Now find L(chi mod 5) zeros up to T=150.
t0 = time.time()
mp.mp.dps = 50

# Need finer step for high t since zeros density increases
# Density for L(chi mod 5): N(T) ~ (T/2pi) log(5T/(2pi e))
# At T=150: N ~ 150/(2pi) * log(5*150/(2pi e)) ~ 23.87 * log(43.85) ~ 23.87 * 3.78 ~ 90
# So ~90 zeros up to T=150. Step 0.3 should be fine.

# Use a slightly smaller dps during scan to speed up; refine at full dps.
def find_zeros_up_to(Z_func, T_max, step=mp.mpf("0.3"), dps_scan=25, dps_refine=50, t_start=mp.mpf("0.1")):
 zeros = []
 mp.mp.dps = dps_scan
 t_prev = mp.mpf(t_start)
 z_prev = Z_func(t_prev)
 t = t_prev + step
 brackets = []
 while t <= T_max:
 z_cur = Z_func(t)
 if z_prev * z_cur < 0:
 brackets.append((t_prev, t))
 t_prev, z_prev = t, z_cur
 t += step
 # Refine at full dps
 mp.mp.dps = dps_refine
 for (a, b) in brackets:
 try:
 r = mp.findroot(Z_func, (a, b), solver='anderson')
 zeros.append(r)
 except Exception as e:
 try:
 r = mp.findroot(Z_func, (a+b)/2)
 zeros.append(r)
 except:
 print(f"Failed to refine [{a},{b}]")
 return zeros

t0 = time.time()
Lchi_zeros_short = find_zeros_up_to(Z_chi, mp.mpf(150), step=mp.mpf("0.3"))
print(f"Found {len(Lchi_zeros_short)} L(chi) zeros up to T=150 in {time.time()-t0:.2f}s")
print(f"First: {Lchi_zeros_short[0]}, Last: {Lchi_zeros_short[-1]}")

Found 90 L(chi) zeros up to T=150 in 30.51s
First: 6.183578195450853914377517309708692525921500579228, Last: 149.34681574873999596109077821703598125293093302889


In [13]:
# Check that the first zero matches LMFDB reference (~6.183578195450...)
# Now L(Delta, s). Analytically normalized so critical line is Re(s)=1/2.
# L(Delta, s) (classical) = Σ τ(n) n^{-s}, with functional equation s -> 12-s.
# Analytic normalization: L_an(Delta, s) = L(Delta, s + 11/2).
# Then critical line is Re(s) = 1/2.
# Coefficients: a_n / sqrt(n^11) where a_n = τ(n).
# So a_n_normalized = τ(n) / n^{11/2}.
# Completed: Λ_an(s) = (2π)^{-s} Γ(s + 11/2) L_an(Delta, s) ... wait let me redo.
#
# Actually: Lambda(s) = (2π)^{-s} Γ(s) L(Δ, s) with L(Δ,s) = Σ τ(n)/n^s (classical), s.t. functional 
# eqn Lambda(s) = Lambda(12-s). So critical line Re(s) = 6.
# Analytic normalization: L_an(s) = L(Δ, s + 11/2). Critical line Re(s_an) = 1/2.
# Completed: Λ_an(s) = (2π)^{-s} Γ(s + 11/2) L_an(s) [check: Λ(s+11/2) = (2π)^{-s-11/2} Γ(s+11/2) L(Δ, s+11/2)].
# Multiplying by overall constant: Λ_an(s) = (2π)^{-s} Γ(s + 11/2) L_an(s) up to a constant.
# Functional equation: Λ_an(s) = Λ_an(1-s) (self-dual).

# Computing L_an(s) via Dirichlet series. We need τ(n). 
# mpmath has Ramanujan tau? Let's check.
try:
 print(mp.taupowmod)
except:
 pass

# Use sympy's symbolic ramanujan_tau, or compute from eta.
# Actually we can compute tau via the product formula:
# Delta(τ) = q ∏_{n≥1} (1-q^n)^24, so τ(n) is the coefficient of q^n in q*∏(1-q^n)^24.
# We can compute the first N values of τ via Euler product.

# Compute τ(1..N) via series multiplication.
def compute_tau_up_to(N):
 # eta(q)^24 = q ∏ (1-q^n)^24
 # Compute via successive multiplication
 # tau values: tau(1)=1, tau(2)=-24, tau(3)=252, tau(4)=-1472, tau(5)=4830, ...
 coeffs = np.zeros(N+1, dtype=object)
 coeffs[0] = 1 # representing series 1 + 0*q + ...
 # We multiply by (1-q^n)^24 successively. Naive but works for small N.
 # Better: use the recursive formula via Newton's identities or use sympy.
 from sympy import Symbol, series, prod, expand
 # That's too slow. Use direct approach:
 # coefficients of ∏_{n=1}^M (1-q^n)^24 up to q^N
 poly = [0]*(N+1)
 poly[0] = 1
 for n in range(1, N+1):
 # Multiply poly by (1-q^n)^24
 # (1-q^n)^24 has coefficients (-1)^k C(24,k) at q^{n*k}, k=0..24
 from math import comb
 binom = [(-1)**k * comb(24, k) for k in range(25)]
 new_poly = [0]*(N+1)
 for i in range(N+1):
 if poly[i] == 0:
 continue
 for k in range(25):
 idx = i + n*k
 if idx > N:
 break
 new_poly[idx] += poly[i] * binom[k]
 poly = new_poly
 # Now poly = ∏(1-q^n)^24 up to q^N. Multiplied by q gives Delta = Σ τ(n) q^n.
 # So τ(n) = poly[n-1].
 tau = [0]*(N+1) # τ[0] unused
 for n in range(1, N+1):
 tau[n] = poly[n-1]
 return tau

# Test
t0 = time.time()
tau_test = compute_tau_up_to(20)
print(f"Computed τ(1..20) in {time.time()-t0:.2f}s")
for n in range(1, 21):
 print(f"τ({n}) = {tau_test[n]}")

Computed τ(1..20) in 1.03s
τ(1) = 1
τ(2) = -24
τ(3) = 252
τ(4) = -1472
τ(5) = 4830
τ(6) = -6048
τ(7) = -16744
τ(8) = 84480
τ(9) = -113643
τ(10) = -115920
τ(11) = 534612
τ(12) = -370944
τ(13) = -577738
τ(14) = 401856
τ(15) = 1217160
τ(16) = 987136
τ(17) = -6905934
τ(18) = 2727432
τ(19) = 10661420
τ(20) = -7109760


In [14]:
# Perfect, matches known values. The O(N^2 * 25) algorithm scales as N^2.
# For X = 10^5 we need tau(n) for all n up to 10^5. That's N^2 * 25 = 10^10 / 25 ops = 2.5e11.
# Way too slow.
#
# Better: For the explicit formula we need Σ_p Σ_k a_p^k * (log p) / p^{k/2} g(k log p).
# We can use the multiplicativity of tau: it suffices to compute tau(p) and tau(p^2) for primes p <= 10^5.
# Then a_p(normalized) = tau(p)/p^{11/2}, and we use Hecke relations:
# a(p^k) = a(p) a(p^{k-1}) - a(p^{k-2}) (for weight 12, level 1: actually a(p^{k+1}) = a_p a(p^k) - p^{11} a(p^{k-1})
# in classical normalization, but in analytic normalization: α(p^{k+1}) = α_p α(p^k) - α(p^{k-1})).

# Since X=10^5, we need primes up to 10^5. For each, we need tau(p). Computing tau(p) for individual 
# primes is faster than the full series.
# Actually, for the EXPLICIT FORMULA arithmetic side, the prime sum involves Σ_p Σ_k a_n Λ(n)/n^{s} g(k log p).
# Wait — for a general L-function L(s,π) with Dirichlet series Σ a(n)/n^s, the explicit formula has
# Σ_n c_n (Λ(n)/sqrt(n)) g(log n) where c_n is related to the Satake parameters.
# Specifically, log L(s,π) = Σ_p Σ_k (1/k) (α_p^k + β_p^k) / p^{ks},
# Differentiating: -L'/L (s,π) = Σ_p Σ_k (α_p^k + β_p^k) log(p) / p^{ks}
# Explicit formula: ... - Σ_p Σ_k (α_p^k + β_p^k) log(p) / p^{k/2} g(k log p).

# For L(Δ) analytically normalized: α_p β_p = 1, α_p + β_p = τ(p)/p^{11/2}.
# (α_p^k + β_p^k) follows from Chebyshev-like recursion.
# So we need primes ≤ X=10^5 and τ(p) for each.

# Computing τ(p) for prime p: use the eta product expansion.
# More efficient: use sympy's algorithm or just compute Σ_{n ≤ p} contribution.
# Actually, the bottleneck for us: for each prime p ≤ 10^5, compute τ(p).
# 
# Use the formula τ(p) = ... related to Eisenstein E_{12}.
# Or use Sage. Or sympy.ntheory: sympy has sympy.ntheory.factor_.tau but that's number of divisors,
# not Ramanujan tau.

# Fastest: use the recursive eta expansion to compute τ(n) for all n ≤ N in O(N^{1.5}) time via
# the pentagonal number theorem then 24th power.
# η(q) = q^{1/24} ∏(1-q^n) = q^{1/24} Σ_k (-1)^k q^{k(3k-1)/2} (pentagonal numbers)
# So ∏(1-q^n) = Σ_k (-1)^k q^{k(3k-1)/2}.
# Then we need this series to the 24th power.

# For brevity, let's use sympy's known function or compute via Hecke.
# Actually, the FASTEST way is to use mpmath's mp.eulerprod or built-in.

# Check if sympy has ramanujan tau:
import sympy.ntheory
# Yes! sympy.ntheory.modular has ... 
# Hmm let me check.
import inspect
funcs = [f for f in dir(sympy) if 'tau' in f.lower() or 'ramanujan' in f.lower()]
print(funcs)

[]


In [15]:
# Compute τ(n) for n up to N using pentagonal number method
# η(q) = Σ_k (-1)^k q^{k(3k-1)/2} for k = ..., -2, -1, 0, 1, 2, ...
# So ∏(1-q^n) = pentagonal series.
# We need (∏(1-q^n))^24 truncated. Use repeated squaring (5 doublings: 1, 2, 4, 8, 16) plus 8 more 
# multiplications: 24 = 16 + 8 = 2^4 + 2^3. So 5 squarings to get to 16, multiply by 8 (also 3 squarings).
# Better: 24 in binary = 11000, so result = base^16 * base^8. Squarings: base^2, base^4, base^8, base^16.
# 4 squarings + 1 multiplication = 5 polynomial multiplications.
# Each poly mult is O(N^2) naively, but ~ O(N log N) with FFT.
# 
# For N=10^5, each O(N^2) mult is 10^10 ops, way too slow.
# We need FFT. Use numpy fft for big integers - but we need exact integer coefficients which can be huge.
# 
# Alternative: For our explicit formula prime sum, we ONLY need τ(p) for primes p up to 10^5 
# (and τ(p^2) for p^2 up to 10^5, i.e., p up to ~316; and τ(p^3) for p ≤ 46, etc).
# So we need τ(n) for n PRIME up to 10^5. That's ~9592 primes.
#
# But computing τ(p) requires the FULL series expansion up to q^p. Different approach.
# 
# Alternative: directly compute τ(p) for each prime by recursive Hecke from Δ definition?
# Or just use the eta-product method to compute τ for all n up to 10^5, but with FFT.

# Idea: Use floating-point FFT to compute the polynomial multiplications.
# τ(n) can be as large as ~ n^{11/2} (Ramanujan-Deligne). For n=10^5, τ(n) up to ~ 10^{27.5}.
# That's way beyond double-precision FFT. Need either:
# - Integer FFT (number theoretic transform), or
# - Split into segments using CRT.
#
# This is getting complex. Let me sidestep: 
# 
# For X = 10^5, we need τ(p^k) for p^k ≤ 10^5. Let's use a more clever route.
# 
# Use sage-like approach via sympy:
# Actually mpmath has nothing for tau directly.
# 
# Let me write a careful eta-pentagonal expansion using PYTHON BIG INTEGERS (slow but exact).
# 
# Actually, let me just use the direct expansion in chunks. The bottleneck is one big poly multiplication.
# Alternative: precompute Δ using ∏(1-q^n)^24 directly via Newton-like recursion. The series
# f(q) = ∏(1-q^n)^24 satisfies a recurrence:
# Take log: log f = 24 Σ log(1-q^n) = -24 Σ_n Σ_k q^{nk}/k.
# So q*f'/f = -24 Σ_n Σ_k n q^{nk}.
# Coefficient of q^m in q*f'/f = -24 σ(m), where σ(m) = Σ d|m of d (divisor sum).
# So m * f_m = (q f'/f)_m * f_0 + Σ_{j<m} (q f'/f)_{m-j} f_j ... 
# More carefully:
# Let f(q) = Σ a_n q^n with a_0 = 1.
# q f'(q) = Σ n a_n q^n.
# We have q f'/f = -24 Σ_{n≥1} σ(n) q^n =: c(q).
# So q f' = c(q) f(q).
# Coefficient of q^m: m a_m = Σ_{k=1}^m c_k a_{m-k}, c_k = -24 σ(k).
# So a_m = (-24/m) Σ_{k=1}^m σ(k) a_{m-k}.
# This is O(N^2) which for N=10^5 is 10^{10}. Still slow but feasible? In Python no.
# 
# Hmm. Let me make a strategic decision:
# X=10^5 is a LARGE cutoff. Computing it requires non-trivial infrastructure.
# 
# I will use a REDUCED prime cutoff for L(Δ) initially (X=10^3 or 10^4) and document this 
# as a bottleneck. The spec says X=10^3 leaves α≈21 baseline; X=10^5 suppresses it to 10^-27 margin.
# For tr-identity verification, the trace residual sensitivity to X depends on test function width.
# At T0=85.7, σ=2, J=10, the test functions are concentrated around T0=85.7 in zero-space.
# The prime sum has terms (log p)/p^{k/2} * g(k log p) where g is FT of test func.
# For Hermite-Gauss centered at T0=85.7 with width σ=2, FT is centered at 0 in u-space with width 1/σ=0.5,
# modulated by exp(i T0 u). So g(k log p) ~ g(0)-like for u = k log p near 0, but actually for k log p > 0 
# we have suppression by Gaussian width 1/σ=0.5 around u=0... wait, FT(Gaussian centered at T0)(u) = exp(i T0 u) * Gaussian(u).
# So |g(u)| ~ exp(-u²σ²/2). For σ=2, |g(u)| ~ exp(-2 u²). At u = log 2 ~ 0.69, |g(u)| ~ exp(-0.96) ~ 0.38.
# At u = log 10 ~ 2.3, |g(u)| ~ exp(-10.6) ~ 2.5e-5.
# At u = log(10^5) ~ 11.5, |g(u)| ~ exp(-265) ~ 10^{-115}.
# So beyond p ~ 10^2 or 10^3, the prime contribution is already tiny.
# 
# Therefore: for the trace identity check at T0=85.7, σ=2, J=10, X=1000 should be MORE than sufficient
# to achieve 10^{-15} precision. Going up to X=10^5 only matters if you want overall scale guarantees.

# DECISION: Use X=10^5 for primes but only need to compute coefficients for primes up to ~ 50 or so 
# for L(Δ) (Hecke higher powers). Primes p where p^k for some k might contribute significantly.
# Actually X is the cutoff on the integer n = p^k. So we need ALL n = p^k ≤ 10^5 (primes p ≤ 10^5, 
# p^2 ≤ 10^5 means p ≤ 316, etc.).
# But the Gaussian suppression makes most contributions negligible.
# 
# OK, let me just compute τ(p) for primes up to some cutoff. With Gaussian suppression so strong, 
# we need only primes p such that |g(log p)| > say 10^{-30}, which means log p ≤ sqrt(30 / (σ²/2)) = sqrt(15) ~ 3.87, 
# p ≤ 48. (For σ=2; with σ²/2 = 2, condition is u² < 15, u < 3.87.)
# 
# But the spec mandates X=10^5. We should include all primes ≤ X, but their contribution will be 
# negligible. So computationally we need only iterate over primes ≤ 10^5 and compute log p and 
# coefficient — for ζ: just need primes (don't need τ). For L(χ): need χ(p^k) easy. For L(Δ): need τ(p^k).
# 
# Let me compute τ(p) for primes up to ~100, which more than covers the contributing range.
# Then I'll iterate the prime loop only up to a cutoff where Gaussian suppression makes terms < 1e-30.

# For now: compute τ for all n ≤ N_tau using the O(N^2) recursion. With N=200, ~40000 ops. Fast.

from sympy import divisor_sigma
def compute_tau_recursive(N):
 """Compute τ(1..N) using q f'/f = -24 Σ σ(n) q^n."""
 sigma_vals = [0] + [int(divisor_sigma(n, 1)) for n in range(1, N+1)]
 c = [0] + [-24 * sigma_vals[k] for k in range(1, N+1)]
 a = [0]*(N+1) # f(q) = Σ a_n q^n, a_0 = 1
 a[0] = 1
 for m in range(1, N+1):
 s = sum(c[k] * a[m-k] for k in range(1, m+1))
 # m * a[m] = s
 assert s % m == 0
 a[m] = s // m
 # f(q) = ∏(1-q^n)^24 = Σ a_n q^n. Δ(q) = q*f(q), so τ(n) = a[n-1].
 tau = [0]*(N+1)
 for n in range(1, N+1):
 tau[n] = a[n-1]
 return tau

t0 = time.time()
tau_full = compute_tau_recursive(200)
print(f"Computed τ(1..200) in {time.time()-t0:.2f}s")
# Verify against known values
known = {1:1, 2:-24, 3:252, 4:-1472, 5:4830, 10:-115920, 11:534612, 100:-1747591019120}
for n, v in known.items():
 print(f"τ({n}): computed={tau_full[n]}, known={v}, {'OK' if tau_full[n]==v else 'MISMATCH'}")

Computed τ(1..200) in 0.01s
τ(1): computed=1, known=1, OK
τ(2): computed=-24, known=-24, OK
τ(3): computed=252, known=252, OK
τ(4): computed=-1472, known=-1472, OK
τ(5): computed=4830, known=4830, OK
τ(10): computed=-115920, known=-115920, OK
τ(11): computed=534612, known=534612, OK
τ(100): computed=37534859200, known=-1747591019120, MISMATCH


In [16]:
# 100th is wrong because we tested only N=200 — wait, it's wrong. Let me verify.
# Actually maybe my "known" value is wrong. Let me check: τ(100)?
# Looking at OEIS A000594: 1, -24, 252, -1472, 4830, -6048, -16744, 84480, -113643, -115920, ...
# τ(11) = 534612, τ(12) = -370944, τ(13) = -577738, ...
# Let me see τ(100). Computed 37534859200. From OEIS table τ(100) = -8497519611648 ? Need to verify.
# Actually OEIS A000594 lists τ(n). Index n=100: 
# Hmm we got 37534859200. Let me cross-check by computing τ(50) and comparing with OEIS.
# OEIS τ(50) = -68496960 (n=50)? Actually let me just trust the recursion and verify by Hecke relation.
# τ is multiplicative: τ(mn) = τ(m)τ(n) for gcd(m,n)=1.
# τ(100) = τ(4) * τ(25) since gcd(4,25)=1.
# Hecke for p=5: τ(5^2) = τ(5)*τ(5) - 5^{11} τ(1) = 4830^2 - 5^{11} = 23328900 - 48828125 = -25499225.
print("τ(25) computed:", tau_full[25])
print("τ(25) from Hecke:", 4830**2 - 5**11)
print("τ(4) computed:", tau_full[4])
print("τ(100) via mult:", tau_full[4] * tau_full[25])

τ(25) computed: -25499225
τ(25) from Hecke: -25499225
τ(4) computed: -1472
τ(100) via mult: 37534859200


In [17]:
# OK so my computed τ(100) = 37534859200 IS correct, my "known" value was wrong.
# So the recursion works perfectly. Now let me compute τ(n) for larger n.

# We only need n = p^k with n ≤ X = 10^5. But due to Gaussian suppression at σ=2, only small primes contribute.
# To be safe, let me compute τ(n) for n ≤ 10^5 via the recursion. Time?
# Recursion: m-th step does m multiply-adds. Total ≈ N^2/2 = 5e9 ops. Too slow in Python.
# 
# Better: only need τ(n) at prime powers p^k. So compute τ(p^k) using:
# - τ(p) computed for each prime via recursion truncated to N=p
# - Hecke: τ(p^{k+1}) = τ(p) τ(p^k) - p^11 τ(p^{k-1})
# 
# But computing τ(p) via the recursion still needs all τ(1..p). So same cost.
# 
# Alternative: Compute τ(1..N) via FFT for large N. Use floating point FFT with care.
# 
# Actually given Gaussian suppression, I'll cap τ computation at N=10^4 (factor 10 from X).
# Beyond N=10^4, contribution to explicit formula is exp(-2*(log 10^4)^2) ~ exp(-170) ~ 10^{-74}.
# So X_effective for L(Δ) at σ=2 = 10^4 is plenty for 10^{-15} precision.
# We compute τ(n) for n ≤ 10^4: 10^4 * 10^4 / 2 = 5e7 ops. Feasible.

t0 = time.time()
N_tau = 10000
tau_arr = compute_tau_recursive(N_tau)
print(f"Computed τ(1..{N_tau}) in {time.time()-t0:.2f}s")
print(f"τ(10000) = {tau_arr[10000]}")

Computed τ(1..10000) in 6.23s
τ(10000) = -482606811957501440000


In [18]:
# Good. Now let's find L(Δ) zeros. Z-function for L(Δ) analytically normalized.
# L_an(Δ, s) = Σ τ(n) n^{-(s + 11/2)} for Re(s) > 1.
# Critical line Re(s) = 1/2. Completed Λ(s) = (2π)^{-s} Γ(s + 11/2) L_an(s).
# Functional equation Λ(s) = Λ(1-s) (Δ is self-dual, root number +1).
# 
# Hardy Z: Z(t) = G(t)/|G(t)| * L_an(1/2 + it), where G(t) = (2π)^{-(1/2+it)} Γ((1/2+it) + 11/2) = (2π)^{-(1/2+it)} Γ(6 + it).
# Since self-dual, eps = 1 and Z(t) is real.

# Compute L_an(Δ, s) for Re(s) = 1/2 using the Dirichlet series with sufficient terms.
# At Re(s) = 1/2, L(Δ, s + 11/2) converges absolutely for Re(s + 11/2) > 13/2, i.e., Re(s) > 1. So at 
# Re(s)=1/2 we are inside the critical strip — need analytic continuation. mpmath can use Euler-Maclaurin.
#
# Better: use approximate functional equation. For now, let's use a smoothed Dirichlet series.
# Or: use mpmath's mp.nsum which can handle.
# Actually a clean approach: use the integral representation via Mellin transform / theta series.

# Simplest robust method: use APPROXIMATE FUNCTIONAL EQUATION.
# Λ(s) = (2π)^{-s} Γ(s + 11/2) L(s) = Λ(1-s).
# Standard AFE:
# L(s) = Σ_n τ(n)/n^{s+11/2} * f(n/X_eff, s) + (functional-eqn-side) Σ_n τ(n)/n^{(1-s)+11/2} * f_tilde(n*X_eff, s)
# where f is a smoothing function.
#
# Simpler: a Riemann-Siegel-like formula for general GL(2) L-functions exists, but complex.
#
# Let me use mpmath's built-in capacities: mp.summation might not work. Try mp.nsum with Dirichlet series 
# and analytic continuation.

# Actually, the cleanest approach: define L(s) as Euler product + use functional equation.
# OR: just use mpmath's `mp.sumem` (Euler-Maclaurin) for analytic continuation of the Dirichlet series.
# 
# Let me try a direct approach using mpmath's mp.sumap or just by AFE with theta smoothing.

# Smoothed AFE: 
# L(s) Γ(s + 11/2) (2π)^{-s} = Σ_n τ(n) f_s(2π n / X) + Σ_n τ(n) f_{1-s}(2π n X) (using G(s) = (2π)^{-s} Γ(s+11/2))
# where f_s(y) = (1/Γ(s+11/2)) Γ(s+11/2, y) (incomplete gamma).
# So L_an(s) = (1/G(s)) [Σ_n τ(n) Γ(s+11/2, 2π n /X) + Σ_n τ(n) Γ(1-s+11/2, 2π n X)] / Γ(s+11/2)
# This needs care.

# Alternative cleaner formulation (Booker, Cohen, etc.):
# For L(s) with functional equation Λ(s) = Λ(1-s), Λ(s) = γ(s) L(s):
# γ(s) L(s) = Σ_n a(n) F_s(n) + Σ_n a(n) F_{1-s}(n)
# where F_s(n) = (1/2πi) ∫_{(2)} γ(s+w) X^w n^{-s-w} dw/w.
# 
# This is getting tedious. Let me just use mpmath's mp.findroot on Z(t) computed via a simpler route.

# Simplest: use a HEURISTIC for finding L(Δ) zeros.
# First zero: γ_1 = 9.2223793999211025 (given in spec).
# 
# Plan: compute L_an(1/2 + it) using mpmath's smoothed AFE. Actually mpmath has lerchphi etc but no L(Δ).
# Let me just use a truncated Dirichlet series with a smooth cutoff (Riemann-Siegel-like).

# For our PURPOSE: we just need zeros of L(Δ) up to T=150 or so. There are limited many; we can find 
# them via root-finding given a quick evaluator.
# 
# Time strategy: implementing L(Δ) evaluation correctly is non-trivial. Let me use mpmath's 
# `mp.lerchphi` no, not the right tool. Use approximate functional equation with smooth cutoff.
# 
# Actually the most pragmatic solution: use known LMFDB zeros for L(Δ).
# LMFDB lists zeros of L(Δ) at https://www.lmfdb.org/L/ModularForm/GL2/Q/holomorphic/1/12/a/a/
# I can hard-code a small list of first several known zeros. But that risks being "fabricated data" 
# in the sense of not being computed from spec. The spec also says "PARI/GP lfunzeros or mpmath".

# Let me try a smoothed AFE approach to compute L(Δ).

# AFE for L_an(Δ, s):
# Λ_an(s) = (2π)^{-s} Γ(s + 11/2) L_an(s) = Λ_an(1-s)
# So L_an(s) = (2π)^s / Γ(s+11/2) Λ_an(s)
# Smoothed AFE (standard form):
# Λ(s) = Σ_n a(n) V_s(n / Y) + Σ_n a(n) V_{1-s}(n Y) (with Y any positive)
# where V_s(y) = (1/2πi) ∫_{(c)} γ(s+u) y^{-u} du/u, γ(s) = (2π)^{-s} Γ(s+11/2).
# Choosing Y=1: Λ(s) = Σ_n τ(n) V_s(n) + Σ_n τ(n) V_{1-s}(n).
# At s = 1/2 + it: Λ(1/2+it) = 2 Re( Σ_n τ(n) V_{1/2+it}(n) ) if Λ self-dual & real-valued on critical line.
# 
# Z(t) = Λ(1/2+it)/|γ(1/2+it)| (theta correction).
# 
# Let's just compute via direct mpmath integration. For each evaluation we need V_s(y) which is an 
# integral. Expensive but workable.

# Actually a cleaner version: V_s(y) for our gamma factor γ(s) = (2π)^{-s} Γ(s + 11/2) equals
# V_s(y) = (1/y^{??}) * Γ(s + 11/2, 2π y) / Γ(s + 11/2) * (2π y)^{?} ... 
# This is just an incomplete Gamma function in disguise.
#
# Recall the integral representation:
# (2π)^{-s} Γ(s + 11/2) = ∫_0^∞ y^{s + 11/2} e^{-2π y} dy/y (with substitution).
# So Λ(s) = ∫_0^∞ y^{s + 11/2 - 1} Σ τ(n) e^{-2π n y} dy / (n^s shift)... wait this is the Mellin transform 
# of Δ: 
# (2π)^{-s} Γ(s) L(Δ, s) = ∫_0^∞ Δ(iy) y^{s-1} dy (in classical normalization, where Δ(iy) = Σ τ(n) e^{-2π n y})
# Analytically normalized: replace s by s + 11/2: 
# (2π)^{-(s+11/2)} Γ(s+11/2) L(Δ, s+11/2) = ∫_0^∞ Δ(iy) y^{s+11/2-1} dy
# So in our normalization: Λ_an(s) (defined as (2π)^{-s} Γ(s+11/2) L_an(s)) = ?
# 
# Let me re-define: L_an(s) := L(Δ, s + 11/2). Then completed (with our chosen Λ_an):
# Λ_an(s) := (2π)^{-s} Γ(s + 11/2) L_an(s) = (2π)^{-s} Γ(s+11/2) L(Δ, s+11/2)
# Compare with classical: Λ(σ) = (2π)^{-σ} Γ(σ) L(Δ, σ). Set σ = s + 11/2:
# Λ(σ) = (2π)^{-(s+11/2)} Γ(s+11/2) L(Δ, s+11/2) = (2π)^{-11/2} * Λ_an(s)
# So Λ_an(s) = (2π)^{11/2} Λ(s+11/2) = (2π)^{11/2} Λ(12 - (s+11/2)) = (2π)^{11/2} Λ(13/2 - s) = (2π)^{11/2} Λ((11/2) + (1-s))
# = (2π)^{11/2} * (2π)^{-11/2} Λ_an(1-s) = Λ_an(1-s). 
# 
# Mellin: Λ_an(s) = (2π)^{11/2} Λ(s+11/2) = (2π)^{11/2} ∫_0^∞ Δ(iy) y^{s+11/2 - 1} dy
# Splitting at y=1: 
# Λ_an(s) = (2π)^{11/2} [∫_0^1 + ∫_1^∞] Δ(iy) y^{s + 11/2 - 1} dy
# Use Δ(iy) = y^{-12} Δ(i/y) (functional equation):
# ∫_0^1 Δ(iy) y^{s+11/2-1} dy = ∫_∞^1 Δ(i/u) u^{12} (1/u)^{s+11/2-1} (-du/u²) (sub y = 1/u)
# = ∫_1^∞ Δ(i/u) u^{12 - (s+11/2-1) - 2} du = ∫_1^∞ Δ(i/u) u^{12 - s - 11/2 + 1 - 2} du
# = ∫_1^∞ Δ(i/u) u^{(11/2) - s + 1 - 2} du, wait let me redo.
# y = 1/u, dy = -du/u^2.
# y^{s+11/2-1} = u^{-(s+11/2-1)} = u^{1-s-11/2}.
# Δ(iy) = Δ(i/u) = u^{12} Δ(iu) (since Δ(iy) = (1/y)^{12} Δ(i/y), i.e., setting y → i/u: Δ(i/u) = u^{12} Δ(iu)).
# Wait Δ has weight 12: Δ(-1/τ) = τ^{12} Δ(τ). With τ = iy, -1/τ = i/y. So Δ(i/y) = (iy)^{12} Δ(iy) = -y^{12} Δ(iy) for y>0?
# (iy)^{12} = i^{12} y^{12} = 1 * y^{12}. So Δ(i/y) = y^{12} Δ(iy).
# I.e., Δ(iy) = y^{-12} Δ(i/y).
# 
# So ∫_0^1 Δ(iy) y^{s+11/2-1} dy = ∫_0^1 y^{-12} Δ(i/y) y^{s+11/2-1} dy = ∫_0^1 Δ(i/y) y^{s + 11/2 - 13} dy
# sub y = 1/u: = ∫_∞^1 Δ(iu) u^{-(s+11/2-13)} (-du/u^2) = ∫_1^∞ Δ(iu) u^{13 - s - 11/2 - 2} du = ∫_1^∞ Δ(iu) u^{-s + 11/2 - 1} du
# = ∫_1^∞ Δ(iu) u^{(1-s) + 11/2 - 1} du.
# So Λ_an(s) = (2π)^{11/2} ∫_1^∞ Δ(iy) [y^{s+11/2-1} + y^{(1-s)+11/2-1}] dy.
# 
# This is exponentially convergent and we can compute it numerically.
# Δ(iy) = Σ τ(n) e^{-2π n y}. For y ≥ 1, the series converges rapidly (e^{-2π} ≈ 1.87e-3).
# So we only need τ(n) for moderate n. At y=1, e^{-2π n} for n up to 30 gives ~ e^{-188} ~ 10^{-82}.

# Great, this is computable! Let me implement.
print("Strategy: use exponentially convergent integral for Λ_an(s).")

Strategy: use exponentially convergent integral for Λ_an(s).


In [19]:
# Implement Λ_an(s) via integral.
# Λ_an(s) = (2π)^{11/2} ∫_1^∞ Δ(iy) [y^{s+11/2-1} + y^{-s+11/2-1+1}] dy
# Hmm let me recompute. Set α = s + 11/2 - 1 and β = (1-s) + 11/2 - 1 = 11/2 - s. So:
# Λ_an(s) = (2π)^{11/2} ∫_1^∞ Δ(iy) [y^α + y^β] dy, α + β = 11 - 1 = ... wait
# α = s + 11/2 - 1 = s + 9/2
# β = (1-s) + 11/2 - 1 = 11/2 - s + 1 - 1 = 11/2 - s. Hmm.
# Total α + β = (s+9/2) + (11/2 - s) = 9/2 + 11/2 = 10. OK.
# At critical line s = 1/2 + it: α = 5 + it, β = 5 - it. Symmetric: α and β are complex conjugates 
# if t is real. So y^α + y^β = 2 Re(y^{5+it}) = 2 y^5 cos(t log y). Real! Good.

# At critical line:
# Λ_an(1/2+it) = (2π)^{11/2} ∫_1^∞ Δ(iy) * 2 y^5 cos(t log y) dy
# = 2 (2π)^{11/2} ∫_1^∞ y^5 cos(t log y) Σ_n τ(n) e^{-2π n y} dy
# = 2 (2π)^{11/2} Σ_n τ(n) ∫_1^∞ y^5 cos(t log y) e^{-2π n y} dy

# This is real on the critical line. Z(t) = Λ_an(1/2+it) / γ-factor magnitude.
# γ(s) = (2π)^{-s} Γ(s + 11/2). |γ(1/2 + it)| = |2π|^{-1/2} |Γ(6 + it)|.
# Λ_an(1/2 + it) is real, so we can just take its sign for zero finding.
# So Z(t) ∝ Λ_an(1/2+it), and zero finding via sign changes.

# Actually we don't even need normalization for ZERO FINDING. So:
def Lambda_an_Delta_re(t, N_terms=100, dps=50):
 mp.mp.dps = dps
 t = mp.mpf(t)
 s = mp.mpc(mp.mpf("0.5"), t)
 total = mp.mpf(0)
 for n in range(1, N_terms+1):
 if n > N_tau or tau_arr[n] == 0:
 continue
 # ∫_1^∞ y^5 cos(t log y) e^{-2π n y} dy. Real value.
 # Use mpmath quad
 integrand = lambda y: y**5 * mp.cos(t * mp.log(y)) * mp.exp(-2*mp.pi*n*y)
 I = mp.quad(integrand, [1, mp.inf])
 total += tau_arr[n] * I
 return 2 * (2*mp.pi)**(mp.mpf("11")/2) * total

# Test at known first zero t ≈ 9.222379...
mp.mp.dps = 30
t0 = time.time()
val = Lambda_an_Delta_re(9.2223793999211025, N_terms=30, dps=30)
print(f"Λ_an(1/2 + 9.22*i): {val} (should be ≈ 0). Time: {time.time()-t0:.2f}s")
val2 = Lambda_an_Delta_re(5.0, N_terms=30, dps=30)
print(f"Λ_an(1/2 + 5i): {val2}")
val3 = Lambda_an_Delta_re(15.0, N_terms=30, dps=30)
print(f"Λ_an(1/2 + 15i): {val3}")

Λ_an(1/2 + 9.22*i): -5.44814550231028245907349750274e-18 (should be ≈ 0). Time: 0.11s
Λ_an(1/2 + 5i): 8.64153414369870298559801555649
Λ_an(1/2 + 15i): 0.000351914064715819112989404535221


In [20]:
# Excellent! Zero at t≈9.222 confirmed (value ≈ 5e-18, essentially zero).
# Now let's check this approach scales well. The slow part is computing the integral at each evaluation.
# We need ~50-100 zeros for L(Δ) up to T=150 (density ~ T/(2π) log(T/(2π e))).
# At T=150: roughly 150/2π log(150 / (2π e)) ≈ 23.87 * log(8.78) ≈ 23.87 * 2.17 ≈ 52 zeros.

# Make Z_Delta(t) a fast function. Vectorize over zeros by precomputing terms.
# Each integral ∫_1^∞ y^5 cos(t log y) e^{-2π n y} dy depends on (n, t). We can express:
# ∫_1^∞ y^5 e^{-2π n y} e^{i t log y} dy + c.c. / 2
# = ∫_1^∞ y^{5+it} e^{-2π n y} dy + c.c.
# = Γ(6+it, 2π n) / (2π n)^{6+it} + c.c. (incomplete Gamma)
# This is the closed form!

# Indeed: ∫_0^∞ y^{s-1} e^{-a y} dy = Γ(s) a^{-s}. 
# ∫_1^∞ y^{s-1} e^{-a y} dy = Γ(s, a) a^{-s} -- wait, incomplete gamma Γ(s,x) = ∫_x^∞ y^{s-1} e^{-y} dy.
# Substitute u = a y: ∫_a^∞ (u/a)^{s-1} e^{-u} du/a = a^{-s} ∫_a^∞ u^{s-1} e^{-u} du = a^{-s} Γ(s, a).
# So ∫_1^∞ y^{s-1} e^{-a y} dy with our integrand y^{5+it} = y^{(6+it)-1}, a = 2π n:
# = (2π n)^{-(6+it)} Γ(6+it, 2π n).

# So:
# Λ_an(1/2 + it) = (2π)^{11/2} Σ_n τ(n) [(2π n)^{-(6+it)} Γ(6+it, 2π n) + (2π n)^{-(6-it)} Γ(6-it, 2π n)]
# = 2 (2π)^{11/2} Re[ Σ_n τ(n) (2π n)^{-(6+it)} Γ(6+it, 2π n) ]
# = 2 (2π)^{11/2 - 6 - it} Re[ Σ_n τ(n) n^{-(6+it)} Γ(6+it, 2π n) * (2π)^{0} ] -- wait let me redo
# = 2 (2π)^{11/2} Re[ Σ_n τ(n) (2π n)^{-(6+it)} Γ(6+it, 2π n) ]
# = 2 (2π)^{11/2 - 6} Re[ Σ_n τ(n) (n)^{-(6+it)} * (2π)^{-it} Γ(6+it, 2π n) ]
# = 2 (2π)^{-1/2} Re[ Σ_n τ(n) n^{-(6+it)} (2π)^{-it} Γ(6+it, 2π n) ]

# That's our formula. mpmath has incomplete gamma: mp.gammainc(s, a, b) = ∫_a^b t^{s-1} e^{-t} dt.
# mp.gammainc(s, a, mp.inf) = Γ(s, a).

def Lambda_an_Delta(t, N_terms=50, dps=30):
 mp.mp.dps = dps
 t = mp.mpf(t)
 s_plus = mp.mpc(6, t)
 twopi = 2*mp.pi
 total = mp.mpc(0)
 for n in range(1, N_terms+1):
 if n > N_tau:
 break
 if tau_arr[n] == 0:
 continue
 Gam = mp.gammainc(s_plus, twopi * n, mp.inf) # Γ(6+it, 2π n)
 term = tau_arr[n] * (twopi * n)**(-s_plus) * Gam
 total += term
 val = 2 * (twopi)**(mp.mpf("11")/2) * total.real
 return val

# Test
t0 = time.time()
print("Λ_an(1/2 + 9.2224*i) =", Lambda_an_Delta(mp.mpf("9.2223793999211025"), 30, dps=30))
print(f"Time: {time.time()-t0:.3f}s")
t0 = time.time()
print("Λ_an(1/2 + 14*i) =", Lambda_an_Delta(14, 30, dps=30))
print(f"Time: {time.time()-t0:.3f}s")

Λ_an(1/2 + 9.2224*i) = 3.15040245201750151384566719869e-18
Time: 0.039s
Λ_an(1/2 + 14*i) = 0.000111850328787850258576569825792
Time: 0.037s


In [21]:
# Much faster (0.04s per eval). Now find L(Δ) zeros up to T=150.
# Watch: at large t, we need more N_terms because Γ(6+it, 2π n) decays slower for small n.
# But e^{-2π n} factor still dominates. For n=1: |Γ(6+it, 2π)| ≈ Γ(6+it, 2π). Rough size 100 at large t.
# Safe to use N_terms = 50.

# Also: at large t the integrand oscillates faster — need higher precision potentially. Use dps=30 for scan.

# Need to be careful: density at T=150 is ~52 zeros. Use step 0.5 initially.
# But near T0=85.7 zero density is ~13.6/zero spacing wait spacing = 2π/log(t/2π).
# At t=85.7: spacing 2π/log(85.7/2π) = 6.28/log(13.6) = 6.28/2.61 = 2.4. So step 0.5 might miss.
# Actually density ~ 1/spacing = 1/2.4 = 0.42/unit. At t=150, spacing = 6.28/log(150/2π) = 6.28/3.17 = 1.98.
# Use step 0.4 to be safe.

def Z_Delta(t):
 return Lambda_an_Delta(t, N_terms=40, dps=30)

t0 = time.time()
LDelta_zeros_short = find_zeros_up_to(Z_Delta, mp.mpf(150), step=mp.mpf("0.4"), 
 dps_scan=30, dps_refine=50, t_start=mp.mpf("0.5"))
print(f"Found {len(LDelta_zeros_short)} L(Δ) zeros up to T=150 in {time.time()-t0:.2f}s")
print(f"First few: {LDelta_zeros_short[:5]}")
print(f"Last: {LDelta_zeros_short[-1]}")

Found 67 L(Δ) zeros up to T=150 in 172.99s
First few: [mpf('9.2223793999211025222437671927408962795543947122671338'), mpf('13.907549861392134406446681328786627571143706411042489'), mpf('17.442776978234473313551525121075072231130888402913144'), mpf('19.656513141954961000127281872628066044905154890994598'), mpf('22.33610363720986727568267188975319273897316736732196')]
Last: 144.89999999999999999999999999916191417429319705793


In [22]:
# Note: the last value 144.89999... looks suspicious -- that's the bracket left endpoint, not a true refined zero.
# Likely the refinement failed. Let me look at the last few.
for z in LDelta_zeros_short[-6:]:
 print(z)
# Verify L(Δ) at this point
print("L(Δ) at suspicious zero:", Lambda_an_Delta(LDelta_zeros_short[-1], 40, dps=30))
print("L(Δ) at second-last zero:", Lambda_an_Delta(LDelta_zeros_short[-2], 40, dps=30))

136.06205611038788336672823085561504406901027320122
136.09999999999999999999999999927298578974831552013
142.89999999999999999988216734304901751063951618712
142.89999999999999999999999999931337546809563132457
144.89060631155981226242570990519490950285982464011
144.89999999999999999999999999916191417429319705793
L(Δ) at suspicious zero: -5.75471132606964900301881282069e-32
L(Δ) at second-last zero: 2.83630722111390521030366349798e-33


In [23]:
# The values are at numerical floor (10^-32). These are NOT real zeros — they're artifacts where the function
# rounds to ~0 at our precision. Need higher dps near the boundary OR use a different solver.
# Also: we have duplicates (136.062... and 136.099..., 142.899... appears twice). Looks like the bracket 
# stuck at endpoints.
# 
# Let me identify which are real zeros vs artifacts. Compute residual at increased precision and discard NaN-like.
# Better: re-do the scan more carefully. The issue is that findroot with 'anderson' may have failed.
# Let me use a more robust solver.

# Let me re-implement with bisection-then-findroot.
def find_zeros_up_to_v2(Z_func, T_max, step=mp.mpf("0.3"), dps_scan=25, dps_refine=50, t_start=mp.mpf("0.1")):
 zeros = []
 mp.mp.dps = dps_scan
 t_prev = mp.mpf(t_start)
 z_prev = Z_func(t_prev)
 t = t_prev + step
 brackets = []
 while t <= T_max:
 z_cur = Z_func(t)
 if z_prev * z_cur < 0:
 brackets.append((t_prev, t, z_prev, z_cur))
 t_prev, z_prev = t, z_cur
 t += step
 # Refine at full dps
 mp.mp.dps = dps_refine
 for (a, b, za, zb) in brackets:
 # Use bisection at full precision
 a_, b_ = mp.mpf(a), mp.mpf(b)
 za_ = Z_func(a_)
 zb_ = Z_func(b_)
 if za_ * zb_ >= 0:
 print(f"Bracket [{a_},{b_}] failed at refine precision: Z(a)={za_}, Z(b)={zb_}")
 continue
 # Bisect to narrow, then findroot
 for _ in range(30): # ~30 bisections gets us to ~10^-9
 c = (a_ + b_) / 2
 zc = Z_func(c)
 if za_ * zc < 0:
 b_, zb_ = c, zc
 else:
 a_, za_ = c, zc
 if abs(b_ - a_) < mp.mpf(10)**(-dps_refine + 5):
 break
 zero = (a_ + b_) / 2
 zeros.append(zero)
 return zeros

# Re-find L(Δ) zeros
t0 = time.time()
LDelta_zeros_short = find_zeros_up_to_v2(Z_Delta, mp.mpf(150), step=mp.mpf("0.3"), 
 dps_scan=30, dps_refine=40, t_start=mp.mpf("0.5"))
print(f"Found {len(LDelta_zeros_short)} L(Δ) zeros up to T=150 in {time.time()-t0:.2f}s")
print(f"First: {LDelta_zeros_short[0]}")
print(f"Last: {LDelta_zeros_short[-1]}")
# Verify
for z in LDelta_zeros_short[-5:]:
 print(f"z={z}, |Λ(z)|={abs(Lambda_an_Delta(z, 40, dps=40))}")

Found 71 L(Δ) zeros up to T=150 in 181.69s
First: 9.22237939997576177120208740235
Last: 134.010766601702198386192321778
z=126.894347381731495261192321778, |Λ(z)|=5.415031052694325699127244284455599280563e-42
z=132.799413486523553729057312012304400336, |Λ(z)|=2.648572648622930088056198405623463326458e-42


z=132.805347826471552252769470215479887434, |Λ(z)|=3.045253298110131463315988974959456352446e-42
z=133.8402576394844800233840942388265106577, |Λ(z)|=4.009403656788130279010411619324764671918e-42
z=134.0107666017021983861923217779344490458, |Λ(z)|=1.446027091473875417077020158366770575825e-42


In [24]:
# Got 71 zeros. But notice issues:
# 1) First zero rounded to lower precision (9.222379399975... vs reference 9.222379399921...)
# Because bisection stopped early.
# 2) Around t=132.8, two zeros very close: 132.799 and 132.805. Suspicious!
# 3) Lost precision overall.
# 
# The bisection is interrupted prematurely. Let me fix the algorithm — also at high t, the function 
# changes sign multiple times in our step. We need a smaller step at large t.
# 
# Also: first zero of L(Δ) is at 9.22237939992110252... (reference). Our 9.22237939997... has matched to ~10 digits.
# Stopping criterion in bisection: |b-a| < 10^{-dps+5} = 10^{-35}. But 30 bisections from step 0.3 only narrow 
# to 0.3 * 2^{-30} ≈ 3e-10. So bisection only got 10 digits. Need to use findroot after bisection.

# Let me actually do the simplest most robust thing: increase dps, use findroot('anderson') with proper bracket, 
# but FIRST check at high precision that bracket is valid.

def find_zeros_robust(Z_func, T_max, step=mp.mpf("0.25"), dps_scan=30, dps_refine=50, t_start=mp.mpf("0.1")):
 zeros = []
 mp.mp.dps = dps_scan
 # Adaptive step: density of zeros grows as log(T). Reduce step at higher T.
 t_prev = mp.mpf(t_start)
 z_prev = Z_func(t_prev)
 t = t_prev + step
 brackets = []
 while t <= T_max:
 z_cur = Z_func(t)
 if z_prev * z_cur < 0:
 brackets.append((float(t_prev), float(t)))
 t_prev, z_prev = t, z_cur
 # Reduce step at higher t
 if t > 50:
 step_use = step * mp.mpf("0.7")
 else:
 step_use = step
 t += step_use
 
 # Refine at high precision
 mp.mp.dps = dps_refine
 for (a, b) in brackets:
 try:
 r = mp.findroot(Z_func, (mp.mpf(a), mp.mpf(b)), solver='anderson')
 # Validate
 r = mp.mpf(r)
 if abs(Z_func(r)) < mp.mpf(10)**(-20):
 zeros.append(r)
 else:
 print(f"Spurious refine: r={r}, Z(r)={Z_func(r)}")
 except Exception as e:
 try:
 r = mp.findroot(Z_func, mp.mpf((a+b)/2))
 r = mp.mpf(r)
 if abs(Z_func(r)) < mp.mpf(10)**(-20):
 zeros.append(r)
 except Exception:
 print(f"Failed bracket [{a}, {b}]: {e}")
 return zeros

# I'll switch strategy: given limited time, I'll first check if 71 zeros gives valid trace identity.
# We have:
# zeta_zeros_short: 52 zeros up to T=150
# Lchi_zeros_short: 90 zeros up to T=150
# LDelta_zeros_short: 71 zeros up to T=150 (with some artifacts near boundary, dps lower)
# 
# For T0=85.7, σ=2, J=10, we need zeros that fall within Gaussian envelope around T0.
# Width σ=2, so effective support is ~T0 ± 5σ to ±10σ = 85.7 ± [10, 20]. Zeros outside contribute exp(-50) or less.
# All our zero lists comfortably cover this range.

# Issue with LDelta_zeros_short artifacts near T=130+: those are FAR from T0=85.7 (distance 45+, so contribution exp(-200)).
# So they're irrelevant for our calculation.

# Let me also clean up the duplicate-looking close zeros: 132.799 and 132.805. Both are 47+ from T0=85.7.
# So we can safely truncate the zero list to T < 120 (say). Then artifacts excluded.

# For now, use first ~40-50 zeros (up to T~110) for our calculation.
LDelta_zeros_clean = [z for z in LDelta_zeros_short if z < 120]
print(f"L(Δ) clean zeros (T<120): {len(LDelta_zeros_clean)}")
print(f"Last: {LDelta_zeros_clean[-1]}")
print(f"First: {LDelta_zeros_clean[0]}")

L(Δ) clean zeros (T<120): 63
Last: 117.5761753265280276536941528326976796927
First: 9.222379399975761771202087402345958810535


In [25]:
# OK, but the first L(Δ) zero is 9.22237939997576... while reference is 9.222379399921...
# Diff is in the 11th digit. Reference says first zero is 9.2223793999211025 (matches LMFDB).
# Our value: 9.222379399975761... — disagrees at 11th digit. This is because bisection only ran 30 iters.
# Let me refine all L(Δ) zeros using a precise findroot.

# Actually let me re-find L(Δ) zeros with the robust v2 method but with more bisections and final findroot.
mp.mp.dps = 50

def refine_zero(Z_func, a, b, max_iters=100, target_dps=40):
 """Refine a zero in [a, b] to high precision."""
 a_, b_ = mp.mpf(a), mp.mpf(b)
 za, zb = Z_func(a_), Z_func(b_)
 if za * zb >= 0:
 return None
 for i in range(max_iters):
 c = (a_ + b_) / 2
 zc = Z_func(c)
 if zc == 0:
 return c
 if za * zc < 0:
 b_, zb = c, zc
 else:
 a_, za = c, zc
 if (b_ - a_) < mp.mpf(10)**(-target_dps - 5):
 break
 return (a_ + b_) / 2

# Refine all L(Δ) zeros to 40+ digits
mp.mp.dps = 50
LDelta_zeros_refined = []
for z in LDelta_zeros_clean:
 # Use bracket around z
 half_step = mp.mpf("0.05")
 a, b = z - half_step, z + half_step
 za, zb = Z_Delta(a), Z_Delta(b)
 if za * zb < 0:
 try:
 r = mp.findroot(Z_Delta, (a, b), solver='anderson')
 LDelta_zeros_refined.append(r)
 except Exception:
 r = refine_zero(Z_Delta, a, b, max_iters=200, target_dps=40)
 if r is not None:
 LDelta_zeros_refined.append(r)
 else:
 # Try wider bracket
 a, b = z - mp.mpf("0.1"), z + mp.mpf("0.1")
 za, zb = Z_Delta(a), Z_Delta(b)
 if za * zb < 0:
 try:
 r = mp.findroot(Z_Delta, (a, b), solver='anderson')
 LDelta_zeros_refined.append(r)
 except:
 pass
print(f"Refined {len(LDelta_zeros_refined)} L(Δ) zeros")
print(f"First refined: {LDelta_zeros_refined[0]}")
print(f"Reference: 9.2223793999211025...")

Refined 49 L(Δ) zeros
First refined: 9.22237939992110252224376719274
Reference: 9.2223793999211025...


In [26]:
# Excellent — first zero matches reference perfectly!
# But we only got 49 of 63. Some refinement failed (the bracket [z±0.05] doesn't bracket the zero).
# That's likely because our original z values were imprecise enough that the bracket missed.
# Let me expand and retry the unrefined ones.

original = LDelta_zeros_clean
refined = LDelta_zeros_refined
print(f"Need to recover {len(original) - len(refined)} zeros")

# Re-do scanning more carefully near 50 < t < 120 with finer step
def fine_scan_refine(Z_func, t_start, t_end, step, dps_scan=30, dps_refine=50):
 mp.mp.dps = dps_scan
 zeros_found = []
 t_prev = mp.mpf(t_start)
 z_prev = Z_func(t_prev)
 t = t_prev + step
 brackets = []
 while t <= t_end:
 z_cur = Z_func(t)
 if z_prev * z_cur < 0:
 brackets.append((mp.mpf(t_prev), mp.mpf(t)))
 t_prev, z_prev = t, z_cur
 t += step
 mp.mp.dps = dps_refine
 for (a, b) in brackets:
 try:
 r = mp.findroot(Z_func, (a, b), solver='anderson')
 if abs(Z_func(r)) < mp.mpf(10)**(-25):
 zeros_found.append(r)
 except Exception as e:
 print(f"Failed [{a},{b}]: {e}")
 return zeros_found

# Use even finer step
print("Re-scanning L(Δ) [50, 120] with finer step...")
t0 = time.time()
LDelta_fine = fine_scan_refine(Z_Delta, mp.mpf("0.5"), mp.mpf(120), step=mp.mpf("0.2"), 
 dps_scan=30, dps_refine=40)
print(f"Found {len(LDelta_fine)} zeros in {time.time()-t0:.1f}s")
print(f"First 3: {LDelta_fine[:3]}")

Need to recover 14 zeros
Re-scanning L(Δ) [50, 120] with finer step...


Found 99 zeros in 286.6s
First 3: [mpf('9.2223793999211025222437671927409'), mpf('13.9075498613921344064466813289476'), mpf('17.4427769782344733135515251350221')]


In [27]:
# 99 zeros up to T=120 — that's a lot more than expected. Likely getting spurious double-counted ones?
# Density should be ~T/2π * log(T/2πe) ≈ 120/6.28 * log(120/(6.28*2.72)) = 19.1 * log(7.03) = 19.1*1.95 = 37.3
# So we expect ~37 zeros up to T=120, not 99!
# Step 0.2 is too small near close pairs, but it shouldn't double-count true single zeros.
# Let me check: are there duplicates?

LDelta_sorted = sorted(LDelta_fine)
diffs = [float(LDelta_sorted[i+1] - LDelta_sorted[i]) for i in range(len(LDelta_sorted)-1)]
print(f"Min spacing: {min(diffs):.6e}, Max: {max(diffs):.4f}, Mean: {sum(diffs)/len(diffs):.4f}")
# Show small ones
small = sorted([(d, i) for i, d in enumerate(diffs)])[:10]
for d, i in small:
 print(f"Gap {d:.4e} between zero {i+1}={LDelta_sorted[i]} and zero {i+2}={LDelta_sorted[i+1]}")

Min spacing: 0.000000e+00, Max: 6.4000, Mean: 1.0869
Gap 0.0000e+00 between zero 36=64.2999999999999999999999999997 and zero 37=64.2999999999999999999999999997
Gap 0.0000e+00 between zero 84=101.899999999999999999999999999 and zero 85=101.899999999999999999999999999
Gap 2.9890e-17 between zero 58=75.8999999999999999999999999995 and zero 59=75.9000000000000000298895558305
Gap 1.4409e-16 between zero 78=95.4999999999999998559092887215 and zero 79=95.4999999999999999999999999993
Gap 1.2576e-15 between zero 96=113.899999999999998742416980673 and zero 97=113.900000000000000000000005316
Gap 4.8877e-14 between zero 82=101.49999999999995112331597625 and zero 83=101.499999999999999999999999999
Gap 4.1955e-13 between zero 30=61.6999999999999994984636531833 and zero 31=61.7000000000004190524832635065
Gap 6.7026e-10 between zero 74=90.8999999999999969459415100505 and zero 75=90.9000000006702555040732695749
Gap 2.0518e-08 between zero 34=63.4999999794816195286210515477 and zero 35=63.49999999999999

In [28]:
# Aha! Many duplicates and spurious zeros. The issue: at large t, my evaluator with N_terms=40, dps=30 
# returns values near the precision floor for many points, so sign changes appear spuriously.
# 
# At t > ~60, |Λ(s)| is becoming very small (modular L falls off rapidly along critical line — actually 
# Hardy density goes UP), but precision might be limited.
# Wait, more likely: Λ_an(s) decays exponentially in |t| because of Γ(6+it). |Γ(6+it)| ~ |t|^{5.5} e^{-π|t|/2}.
# So Λ ~ |t|^{5.5} e^{-π|t|/2} * L(1/2+it). At t=100, |t|^{5.5} e^{-π*100/2} = 1e11 * e^{-157} ~ 10^{-57}.
# So Λ(1/2 + 100i) is around 10^{-57}, far below our dps=30 precision.
# 
# We need to use Z(t) = Λ(1/2+it) / |γ(1/2+it)| (Hardy Z normalization).
# |γ(1/2+it)| = (2π)^{-1/2} |Γ(6+it)|, so Z(t) = sqrt(2π) Λ(1/2+it) / |Γ(6+it)|.

# Better Z formulation:
def Z_Delta_proper(t, N_terms=40, dps=30):
 mp.mp.dps = dps
 t = mp.mpf(t)
 Lambda_val = Lambda_an_Delta(t, N_terms=N_terms, dps=dps)
 # Normalize by |γ|: |Γ(6+it)| * (2π)^{-1/2}
 gamma_mag = abs(mp.gamma(mp.mpc(6, t)))
 norm = gamma_mag / mp.sqrt(2*mp.pi)
 if norm < mp.mpf(10)**(-dps + 5):
 return mp.mpf(0)
 return Lambda_val / norm

# Test 
mp.mp.dps = 30
print("Z_Delta(9.22) =", Z_Delta_proper(9.2223793999211025))
print("Z_Delta(50) =", Z_Delta_proper(50))
print("Z_Delta(100) =", Z_Delta_proper(100))
print("Z_Delta(150) =", Z_Delta_proper(150))

Z_Delta(9.22) = -3.91980469247697313730544460894e-17
Z_Delta(50) = 2.79736791175323538251838435138
Z_Delta(100) = 0.0
Z_Delta(150) = 0.0


In [29]:
# OK at t=100 with dps=30 we get 0 because of underflow in Λ. We need higher dps.
# Let me increase dps for evaluation.

mp.mp.dps = 50
print("Z_Delta(100, dps=50) =", Z_Delta_proper(100, N_terms=50, dps=50))
print("Z_Delta(150, dps=50) =", Z_Delta_proper(150, N_terms=50, dps=50))
mp.mp.dps = 80
print("Z_Delta(100, dps=80) =", Z_Delta_proper(100, N_terms=60, dps=80))
print("Z_Delta(150, dps=80) =", Z_Delta_proper(150, N_terms=70, dps=80))

Z_Delta(100, dps=50) = 0.0
Z_Delta(150, dps=50) = 0.0


Z_Delta(100, dps=80) = -0.23390659155684520570658245244816233376205296553590516778831448517371067482858976


Z_Delta(150, dps=80) = 0.0


In [30]:
# So at dps=80 we get reliable values up to t~100. For t=150 we need higher.
# But we only need zeros up to T0 + tail. T0=85.7, σ=2, J=10, max Hermite poly H_9 has degree 9, support 
# extends to ~T0 + 10σ at exp(-50) ≈ 1e-22. So we need zeros up to T ~ 105 ish for 1e-15 tolerance.

# Let's redo L(Δ) zero finding up to T=110 with high precision Z function.
def Z_Delta_safe(t):
 return Z_Delta_proper(t, N_terms=60, dps=60)

mp.mp.dps = 60
# Quick sanity
print("Z_D(50, dps60):", Z_Delta_safe(50))
print("Z_D(110, dps60):", Z_Delta_safe(110))

Z_D(50, dps60): 2.79736775413562335718634360924067265992661347132589471929294


Z_D(110, dps60): 0.0


In [31]:
# dps=60 not enough for t=110. Let me use dps=80.
def Z_Delta_safe(t):
 return Z_Delta_proper(t, N_terms=80, dps=80)

mp.mp.dps = 80
print("Z_D(110):", Z_Delta_safe(110))
print("Z_D(120):", Z_Delta_safe(120))
print("Z_D(105):", Z_Delta_safe(105))

Z_D(110): 2.3664373755063361331312145556758855609458378197542590539248097832733510950589466


Z_D(120): 0.040746689280150732607502757595061439850500755302472300228692223421021126804795169


Z_D(105): -1.4865943765230491733885870030414207999276798695748016178120791894542216773656971


In [32]:
# Good. Now scan for L(Δ) zeros up to T=110 with proper precision.
t0 = time.time()
LDelta_zeros = fine_scan_refine(Z_Delta_safe, mp.mpf("0.5"), mp.mpf(110), step=mp.mpf("0.3"),
 dps_scan=80, dps_refine=80)
print(f"Found {len(LDelta_zeros)} L(Δ) zeros up to T=110 in {time.time()-t0:.1f}s")
print(f"First: {LDelta_zeros[0]}")
print(f"Last: {LDelta_zeros[-1]}")
# Check no duplicates
sorted_L = sorted(LDelta_zeros)
diffs = [float(sorted_L[i+1] - sorted_L[i]) for i in range(len(sorted_L)-1)]
print(f"Min gap: {min(diffs):.4e}, Max: {max(diffs):.4f}")

TimeoutError: Code execution timed out after 900.0 seconds

In [33]:
# Too slow at dps=80 for full scan. Let me use a smarter approach:
# Scan at lower dps to FIND brackets, then refine at higher dps only at brackets.
# Also reduce N_terms when possible.

# We already have brackets from the previous scan (LDelta_fine), but it had spurious double-detections.
# Let me re-do with smarter approach: use dps=40 for scan (good enough where |Λ| > 1e-30), 
# and tag zeros where evaluator returns ~0 (don't count those).

import sys
print("Continuing with smarter strategy.")
print(f"Time used so far roughly: see clock")

# Actually let me check: I already have LDelta_zeros_refined from earlier (49 zeros, good precision, T<120 region).
# Let me look at gaps.
sorted_orig = sorted(LDelta_zeros_refined)
print(f"Length: {len(sorted_orig)}")
diffs = [float(sorted_orig[i+1] - sorted_orig[i]) for i in range(len(sorted_orig)-1)]
print(f"Min gap: {min(diffs):.4f}, Max gap: {max(diffs):.4f}, Mean: {sum(diffs)/len(diffs):.4f}")
print(f"First: {sorted_orig[0]}")
print(f"Last: {sorted_orig[-1]}")

Continuing with smarter strategy.
Time used so far roughly: see clock
Length: 49
Min gap: 0.0070, Max gap: 8.0918, Mean: 2.2584
First: 9.222379399921102522243767192737740835933510665045486921201009500315137756842887
Last: 117.62613182749800743639740073804507538566805485469525971876231240642596276302356


In [34]:
# Max gap is 8.0918 — that's too large. Expected gap ~ 2π/log(T/2πe) which at T=85.7 is ~2.4. 
# So we're missing zeros. Let me get the gap locations:
big_gaps = [(diffs[i], sorted_orig[i], sorted_orig[i+1]) for i in range(len(diffs)) if diffs[i] > 4]
for gap, a, b in big_gaps:
 print(f"Gap {gap:.3f}: {a} → {b}")

Gap 4.685: 9.222379399921102522243767192737740835933510665045486921201009500315137756842887 → 13.907549861392134406446681328865513661665807591583662124103820678655552001146134
Gap 4.035: 61.236700398686477482055675487520847171475511105897100933362353636368879961082712 → 65.271483797796661616079784535137022091385245045599619714490891020375329389935359
Gap 5.050: 65.628679801411891913667655417597288595930287773485433842554637173805076599819586 → 70.678447122978699135896962613301625463101609097444217840247593365177181112812832
Gap 7.529: 72.375502489726513003277055999223712712252617145193976286830783717363146934076212 → 79.904688650136813521385192872148930346823625390918728128922254860810880927601829
Gap 4.068: 81.69027011775515851571403410091888151635755697872707538874370580828099264181219 → 85.758581051671044356670765254181314692504970133469258076683305880294483358738944
Gap 8.092: 85.758581051671044356670765254181314692504970133469258076683305880294483358738944 → 93.8504015467064238900

In [35]:
# Notably, the big gap 85.7 → 93.85 happens right around our target T0=85.7! Missing zeros at around 87-93.
# Big gap of 8.09 means missing 2-3 zeros there. This will break the trace identity.
# Need to find all zeros in these gaps. Let me focus on fixing scan near T0.

# Use targeted scan in big-gap regions with finer step.
mp.mp.dps = 50
def targeted_scan(Z_func, a, b, step, dps_scan=40, dps_refine=50):
 mp.mp.dps = dps_scan
 t_prev = mp.mpf(a)
 z_prev = Z_func(t_prev)
 t = t_prev + step
 brackets = []
 while t <= b:
 z_cur = Z_func(t)
 if z_prev * z_cur < 0:
 brackets.append((mp.mpf(t_prev), mp.mpf(t)))
 t_prev, z_prev = t, z_cur
 t += step
 
 mp.mp.dps = dps_refine
 zeros = []
 for a_, b_ in brackets:
 try:
 r = mp.findroot(Z_func, (a_, b_), solver='anderson')
 if abs(Z_func(r)) < mp.mpf(10)**(-25):
 zeros.append(r)
 except:
 pass
 return zeros

# Use Z_Delta_proper at dps=50 with N_terms=60 — should be fine up to t~95
def Z_Delta_d50(t):
 return Z_Delta_proper(t, N_terms=50, dps=50)

# Scan critical region near T0=85.7
print("Targeted scan: [85.7, 95]")
t0 = time.time()
extra = targeted_scan(Z_Delta_d50, mp.mpf("85.75"), mp.mpf("95"), step=mp.mpf("0.15"))
print(f"Found {len(extra)} in [85.75, 95] in {time.time()-t0:.1f}s")
for z in extra:
 print(f" {z}, |Z|={abs(Z_Delta_d50(z))}")

Targeted scan: [85.7, 95]


Found 0 in [85.75, 95] in 9.9s


In [36]:
# Zero zeros found in [85.75, 95]? Let me check Z values directly.
mp.mp.dps = 50
for t in [86, 87, 88, 89, 90, 91, 92, 93, 94]:
 print(f"Z_Delta({t}) = {Z_Delta_d50(t)}")

Z_Delta(86) = 0.0
Z_Delta(87) = 0.0


Z_Delta(88) = 0.0
Z_Delta(89) = 0.0


Z_Delta(90) = 0.0
Z_Delta(91) = 0.0


Z_Delta(92) = 0.0
Z_Delta(93) = 0.0


Z_Delta(94) = 0.0


In [37]:
# All zero — Λ is underflowing. Need higher dps.
mp.mp.dps = 70
def Z_Delta_d70(t):
 return Z_Delta_proper(t, N_terms=60, dps=70)

for t in [86, 87, 88, 89, 90, 91, 92, 93, 94]:
 v = Z_Delta_d70(t)
 print(f"Z_Delta({t}) = {v}")

Z_Delta(86) = -2.872838703508239954375050844770883685203586672807228604365245318195884


Z_Delta(87) = 1.516889546085128505350059809263293422711837741795208660933541840785477


Z_Delta(88) = 0.3765562010570747059979514628674955144222143014498777995127171539454611


Z_Delta(89) = -0.1016718484551635155288761372790049808922201236027710291493899371560844


Z_Delta(90) = 1.629380090135831598406528688901790132344969656157326469873778789064922


Z_Delta(91) = -0.3167185103951839274624865765787199415208425335927687077756492617687999


Z_Delta(92) = 2.027730516981759326430894289885316030890227795150316983349399873823519


Z_Delta(93) = -2.733913858390484608032430794953896153838907190825782311305624872218167


Z_Delta(94) = 1.409406903380079915968216575353070526895627843263887167516946595899757


In [38]:
# Now we can see sign changes: 86→87, 88→89, 89→90, 90→91, 91→92, 92→93, 93→94.
# Many zeros in this range. Let me scan with dps=70.

mp.mp.dps = 70
def Z_D_safe(t):
 return Z_Delta_proper(t, N_terms=70, dps=70)

t0 = time.time()
print("Scanning [80, 110] at dps=70...")
extra = targeted_scan(Z_D_safe, mp.mpf("80"), mp.mpf("110"), step=mp.mpf("0.2"), 
 dps_scan=70, dps_refine=70)
print(f"Found {len(extra)} zeros in [80,110] in {time.time()-t0:.1f}s")
for z in extra[:20]:
 print(f" {z}")

Scanning [80, 110] at dps=70...


Found 0 zeros in [80,110] in 332.0s


In [39]:
# Time pressure! The L(Δ) zero finding at high dps is too slow.
# Strategy: report partial results immediately.
# The brackets must be found at dps_scan=70 but refined separately. The error message says "0 found" — 
# my targeted_scan refines at dps_refine, but it seems the function may not be passing brackets correctly.
# But no time to debug. Let me cache what I have and assemble a partial validation report.

import pickle, time

mp.mp.dps = 50

# Cache zero lists
artifacts = {}
with open('cache/zeta_zeros_T150.pkl', 'wb') as f:
 pickle.dump([str(z) for z in zeta_zeros_short], f)
with open('cache/Lchi5_zeros_T150.pkl', 'wb') as f:
 pickle.dump([str(z) for z in Lchi_zeros_short], f)
with open('cache/LDelta_zeros_partial.pkl', 'wb') as f:
 pickle.dump([str(z) for z in LDelta_zeros_refined], f)
 
print("Cached partial zero lists")
print(f"ζ: {len(zeta_zeros_short)} zeros up to T=150")
print(f"L(χ₄ mod 5): {len(Lchi_zeros_short)} zeros up to T=150")
print(f"L(Δ): {len(LDelta_zeros_refined)} zeros (partial, has gaps near T0=85.7)")

Cached partial zero lists
ζ: 52 zeros up to T=150
L(χ₄ mod 5): 90 zeros up to T=150
L(Δ): 49 zeros (partial, has gaps near T0=85.7)


In [40]:
# Build the M_zeros and M_arith for ζ and L(χ) and check the trace identity.
# Time-critical: do the simplest possible correct implementation.

# Hermite-Gauss orthonormal basis at T0=85.7, σ=2, J=10
# φ_n(t) = (1/sqrt(σ)) * h_n((t-T0)/σ), where h_n(x) = (1/sqrt(2^n n! sqrt(π))) H_n(x) exp(-x²/2)
# 
# Inner product is over the real line (Plancherel/Hermite-Gauss orthonormal).

# Use mpmath for high precision.
mp.mp.dps = 50

T0 = mp.mpf("85.7")
sigma = mp.mpf("2")
J = 10

# Hermite polynomial H_n(x) - physicists', recursive
def hermite_phys(n, x):
 if n == 0:
 return mp.mpf(1)
 if n == 1:
 return 2*x
 Hm1 = mp.mpf(1)
 Hm = 2*x
 for k in range(1, n):
 Hp = 2*x*Hm - 2*k*Hm1
 Hm1 = Hm
 Hm = Hp
 return Hm

# Normalization factor: 1/sqrt(2^n n! sqrt(π))
def phi_hg_mp(n, x):
 # In terms of x = (t-T0)/sigma
 Hn = hermite_phys(n, x)
 fact = mp.factorial(n)
 norm = 1 / mp.sqrt((mp.mpf(2)**n) * fact * mp.sqrt(mp.pi))
 return norm * Hn * mp.exp(-x*x/2)

def phi_n_t(n, t):
 """Orthonormal Hermite-Gauss in t-space."""
 x = (t - T0)/sigma
 return phi_hg_mp(n, x) / mp.sqrt(sigma)

# Test orthonormality
print("Testing orthonormality:")
for i in range(3):
 for j in range(3):
 val = mp.quad(lambda t: phi_n_t(i, t)*phi_n_t(j, t), [-mp.inf, mp.inf])
 if i == j:
 print(f"<φ_{i},φ_{j}> = {val}")
 elif abs(val) > 1e-30:
 print(f"<φ_{i},φ_{j}> = {val} (should be 0)")

Testing orthonormality:
<φ_0,φ_0> = 4.194281910401093076889982037787623103334310694774e-225
<φ_1,φ_1> = 4.3559012778814614089934587452471272298436889650234e-222
<φ_2,φ_2> = 2.2575203793314549014719491123331330500022410638944e-219


In [41]:
# The quadrature went wrong — it's evaluating at infinity probably. Let me use finite limits and watch.
print(mp.quad(lambda t: phi_n_t(0, t)**2, [T0 - 15*sigma, T0 + 15*sigma]))

1.0


In [42]:
# OK orthonormality is fine with proper integration limits.
# 
# Trace identity check: tr(M_zeros) = Σ_γ Σ_j φ_j(γ)², compared to arithmetic side of Weil EF applied to ψ(t)=Σ_j φ_j(t)².
#
# For TIME REASONS, I'll compute tr(M_zeros) directly from zero list, and tr(M_arith) via Weil EF.

# Compute tr(M_zeros) for ζ:
def psi_func(t):
 """ψ(t) = Σ_{j=0}^{J-1} φ_j(t)²"""
 s = mp.mpf(0)
 for j in range(J):
 s += phi_n_t(j, t)**2
 return s

# Test
print(f"ψ(T0) = {psi_func(T0)}") # should be sum of phi_j(T0)^2
# Check normalization: ∫ψ(t)dt = Σ ∫φ_j² = Σ 1 = J
psi_integral = mp.quad(psi_func, [T0 - 15*sigma, T0 + 15*sigma])
print(f"∫ψ(t)dt = {psi_integral} (should be J={J})")

ψ(T0) = 0.69421765163102824370564463766266939273779667280717


∫ψ(t)dt = 10.0 (should be J=10)


In [43]:
# Good. Now compute tr(M_zeros) for ζ and L(χ).
mp.mp.dps = 50

# ζ zeros - sum ψ(γ) over both ±γ since explicit formula sums over all zeros γ + complex conjugates.
# Hermite-Gauss centered at T0=85.7 > 0; negative zeros (at -γ_ρ) are negligible (>85 distance).
# But for completeness include them.

# We treat zeros as the set of imaginary parts γ_n > 0 and their negatives -γ_n.
# The Weil EF sums over ALL γ (positive and negative).

# For T0 = 85.7 ≫ 0, negative γ contributions are negligible (~ exp(-(2*85.7)^2/8) ~ exp(-3672) ~ 0).
# So we can just sum over positive γ.

def trace_M_zeros(zero_list):
 s = mp.mpf(0)
 for gamma in zero_list:
 s += psi_func(gamma)
 return s

t0 = time.time()
tr_zeros_zeta = trace_M_zeros(zeta_zeros_short)
print(f"tr(M_zeros) for ζ = {tr_zeros_zeta} ({time.time()-t0:.2f}s)")

t0 = time.time()
tr_zeros_Lchi = trace_M_zeros(Lchi_zeros_short)
print(f"tr(M_zeros) for L(χ) = {tr_zeros_Lchi} ({time.time()-t0:.2f}s)")

t0 = time.time()
tr_zeros_LDelta = trace_M_zeros(LDelta_zeros_refined)
print(f"tr(M_zeros) for L(Δ) (partial) = {tr_zeros_LDelta} ({time.time()-t0:.2f}s)")

tr(M_zeros) for ζ = 3.8982947980685309871749953482165299190621046029871 (0.04s)
tr(M_zeros) for L(χ) = 6.6869909943701156025364361702651152406578478097096 (0.05s)
tr(M_zeros) for L(Δ) (partial) = 2.0963123974175338130709515605101539417207065129595 (0.03s)


In [44]:
# Now compute tr(M_arith) via Weil's explicit formula applied to ψ.
# 
# Weil EF (Riemann form): for nice test function h(r) (even, holomorphic):
# Σ_ρ h(γ_ρ) = pole terms + (1/2π) ∫ h(r) (γ-factor logarithmic derivative) dr 
# - 2 Σ_{n≥2} Λ_arith(n)/sqrt(n) g(log n)
# where g(u) = (1/2π) ∫ h(r) e^{-iru} dr is the inverse FT of h.
#
# For ζ:
# Σ h(γ) = h(i/2) + h(-i/2) - g(0) log(π) + (1/2π) ∫ h(r) Re ψ_d(1/4 + ir/2) dr 
# - 2 Σ_{n≥2} Λ(n)/sqrt(n) g(log n)
# where Λ(n) = log p if n=p^k else 0, ψ_d = Γ'/Γ digamma.
#
# For L(χ) primitive mod q, even character (a=0) or odd (a=1):
# Σ h(γ) = - g(0) log(π/q) + (1/2π) ∫ h(r) Re ψ_d((1/2 + a + ir)/2) ...
# Wait, archimedean Γ-factor for L(s,χ): γ_∞(s) = π^{-(s+a)/2} Γ((s+a)/2). So at s = 1/2 + ir:
# γ_∞(s) = π^{-(1/2+a+ir)/2} Γ((1/2+a+ir)/2)
# Log derivative w.r.t. s = -log(π)/2 + (1/2) ψ_d((1/2+a+ir)/2)
# Times 2 for both sides of functional eqn (s and 1-s).
# 
# Use cleaner notation. Let h(r) be test function (even, real on real line).
# The general explicit formula (multiplicative form):
# Σ_γ h(γ) = (poles in original strip via h(i*Im(pole))) 
# + (1/(2π)) ∫_{-∞}^∞ h(r) * [Γ-factor log-derivative on critical line] dr
# - 2 Σ_{p,k} (log p)/p^{k/2} * cos-FT-style * (chi(p^k) or Hecke for L(Δ))
#
# Let me write this cleanly for each.

# Compute g(u) = (1/2π) ∫_{-∞}^∞ h(r) e^{-iru} dr where h = ψ_func.
# But ψ is centered at T0, so h is NOT even. The general EF requires even h.
# 
# Actually we need EVEN test function. ψ(t) = Σ φ_j(t)² is centered at T0, not even.
# To make it even, we should consider the symmetric version: ψ_sym(t) = (ψ(t) + ψ(-t))/2... 
# but the EF naturally handles this via Σ_γ h(γ) where γ ranges over ALL zeros (positive AND negative imaginary parts).

# Actually the Riemann-Weil explicit formula in its full form:
# Σ_{ρ} h(γ_ρ) over all zeros (where ρ = 1/2 + iγ, γ can be positive or negative for non-self-dual).
# For ζ and L(Δ) (self-dual), zeros come in pairs ±γ.
# For L(χ mod 5), non-self-dual; zeros at γ_n for chi correspond to -γ_n for chibar (different L-function!).
# So for L(χ) we sum over γ_n (all real values where L(1/2+iγ_n, χ)=0). They're not symmetric around 0.

# This is getting complex. Given time constraint, let me just compute numerically the EF for ζ as a 
# concrete validation.

# For ζ EF with even test function H(r):
# Σ_γ H(γ) = H(i/2) + H(-i/2) - (1/2π) ∫ H(r) log(π) dr + (1/2π) ∫ H(r) Re[ψ(1/4 + ir/2)] dr 
# - 2 Σ_{n≥2} Λ(n)/sqrt(n) ĥ(log n)
# where ĥ(u) = (1/2π) ∫ H(r) e^{-iru} dr (Fourier transform).
# Sum is over all γ (including negative), with ζ zeros symmetric in ±γ.

# We have non-even ψ centered at T0. Let's just use it: extending the EF to a non-even h means we 
# also need contributions from γ in the lower half-plane (negative γ). For ζ, these are also zeros, 
# but ψ(γ) for γ < 0 is essentially zero (Gaussian decay).
# 
# Effectively: Σ_{γ > 0} ψ(γ) + Σ_{γ < 0} ψ(γ) = arithmetic side. Second sum is ~0.

# Let me define the EF more carefully for ψ.
# Take h(r) = ψ(r). Then:
# Σ_γ h(γ) [where γ runs over all zeros, ±]
# = h(i/2) + h(-i/2) [poles at s=0, s=1]
# - g(0) log π
# + (1/2π) ∫_{-∞}^∞ h(r) Re[ψ_d(1/4 + ir/2)] dr
# - 2 Σ_{n=2}^∞ Λ(n)/sqrt(n) g(log n)
# where g(u) = (1/2π) ∫ h(r) e^{-iru} dr.

# For ψ(t) = Σ φ_j(t)², Fourier transform of φ_j(t)² is convolution of FT(φ_j) with itself.
# FT of phi_j_t (orthonormal at T0,σ): phi_hat_j(u) = (1/sqrt(σ)) e^{-i T0 u} (sigma e^{-σ²u²/2} H_j(σu) (-i)^j ... ) up to factors.
# Easier to compute g(u) numerically by quadrature.

# Note: in our application h(γ) is real (ψ is real). 
# g(u) is complex in general, but g(-u) = conj(g(u)) since h is real.
# g(0) = (1/2π) ∫ h = (1/2π) * J = J/(2π).

print(f"J/(2π) = {J/(2*float(mp.pi))}")
mp.mp.dps = 50
g0 = mp.mpf(J) / (2*mp.pi)
print(f"g(0) = {g0}")

J/(2π) = 1.5915494309189535
g(0) = 1.5915494309189533576888376337251436203445964574046


In [45]:
# Compute the arithmetic side of the Weil explicit formula for ζ.
# Time-constrained: limit precision but get the right structure.

mp.mp.dps = 50

# (1) Pole terms: h(i/2) + h(-i/2). h = ψ. But ψ is defined on real line; for complex arg, use the analytic expression.
# φ_j(t) = (1/sqrt(σ)) * h_j_hg((t-T0)/σ).
# h_j_hg(x) = (1/sqrt(2^j j! sqrt(π))) H_j(x) exp(-x²/2).
# For t = i/2: x = (i/2 - T0)/σ. 
# Since ψ(t) has Gaussian factor exp(-(t-T0)²/σ²), at t=i/2: (t-T0)² = (i/2 - 85.7)² ≈ -1/4 - i*85.7 - 85.7² ≈ -7344.5 - 85.7i.
# exp(-(t-T0)²/σ²) = exp(-(-7344.5 - 85.7i)/4) = exp(1836.1 + 21.4i) — gigantic!
# Wait this doesn't decay because t is complex. The Hermite-Gauss has exp(-x²/2), not |x|².
# 
# For t imaginary, exp(-((t-T0)/σ)²/2) = exp(-positive) for t real, but for t = i/2:
# (t-T0)/σ ≈ (i/2 - 85.7)/2 → real part -85.7/2, imag part 1/4 (small).
# x = -42.85 + 0.25i; x² = 1836 - 21.4i - 0.0625 ≈ 1836 - 21.4i.
# exp(-x²/2) = exp(-918 + 10.7i) ≈ 0 (negligible).
# Wait that's exp(-918), tiny. So h(i/2) is negligible. Good — the pole terms don't dominate.

# Actually: -x²/2 ≈ -(1836 - 21.4i)/2 = -918 + 10.7i. exp = e^{-918} * (cos(10.7) + i sin(10.7)) ≈ 0.
# So h(i/2) ≈ 0. Pole contributions negligible.

# (2) g(0) log π term. log π ≈ 1.1447. g(0) = J/(2π) = 1.5915. Term = - g(0) log π ≈ -1.822.
# (3) Archimedean integral.
# (4) Prime sum.

# Let me just compute everything numerically.

# Archimedean integral I_arch = (1/2π) ∫ h(r) Re[ψ_d(1/4 + ir/2)] dr
# where ψ_d = Γ'/Γ.
# h(r) = ψ_func(r) is concentrated around T0=85.7.
# For r large, Re[ψ_d(1/4 + ir/2)] ≈ log|ir/2| - 1/(4r) ≈ log(r/2). [Stirling]

# Numerical integration:
def arch_integrand_zeta(r):
 r_mp = mp.mpf(r) if not isinstance(r, mp.mpf) else r
 return psi_func(r_mp) * mp.re(mp.digamma(mp.mpf("0.25") + mp.mpc(0, r_mp/2)))

# Effective integration range: ψ centered at T0=85.7 with width σ=2, support T0 ± 12σ = [61.7, 109.7]
t_lo = T0 - 12*sigma
t_hi = T0 + 12*sigma
I_arch = mp.quad(arch_integrand_zeta, [t_lo, t_hi]) / (2*mp.pi)
print(f"I_arch = {I_arch}")
# Also negative half (symmetric ζ)
# Actually the integral is from -∞ to ∞. We also need contribution from t near -T0? Well ψ is concentrated 
# at T0, not -T0, so the integrand is small at negative r. Ignore.

I_arch = 5.9783922212210922281828970530543552361164737571726


In [46]:
# Compute g(u) = (1/2π) ∫ h(r) e^{-iru} dr — Fourier transform.
# For our use: g(log n) for n = p^k.
# h(r) = ψ(r) is concentrated around r=T0 with width σ. 
# Compute g(u) numerically.

def g_func(u):
 """g(u) = (1/2π) ∫ ψ(r) e^{-iru} dr"""
 u_mp = mp.mpf(u) if not isinstance(u, mp.mpf) else u
 integrand_re = lambda r: psi_func(r) * mp.cos(r * u_mp)
 integrand_im = lambda r: -psi_func(r) * mp.sin(r * u_mp)
 re = mp.quad(integrand_re, [t_lo, t_hi]) / (2*mp.pi)
 im = mp.quad(integrand_im, [t_lo, t_hi]) / (2*mp.pi)
 return mp.mpc(re, im)

# Sanity: g(0) should equal J/(2π)
print(f"g(0) numerical = {g_func(0)}")
print(f"g(log 2) = {g_func(mp.log(2))}")
print(f"g(log 1000) = {g_func(mp.log(1000))}")

g(0) numerical = (1.5915494309189533576888376337251436203445964574039 + 0.0j)


g(log 2) = (0.11809296193789714677535471953638749839195264428869 + 0.034925207379897120769798137667539979350921855662783j)


g(log 1000) = (-0.000000000037078972000986607615214063423247863815755103907521 + 0.00000000018715354199300288825628160883387013502098179567253j)


In [47]:
# Excellent! g(u) computed.
# 
# Note: in the EF, we have -2 Σ Λ(n)/sqrt(n) g(log n), but with ALL zeros (positive AND negative γ),
# this matches the EF when h is real but not necessarily even. For the SUM over γ on both sides:
# Σ_{γ real} h(γ) where γ ranges over ALL zero imaginary parts.
# The prime sum is -2 Re Σ_{n} Λ(n)/sqrt(n) g(log n) when h is real (because we count both u and -u contributions).
# 
# Actually, the standard form: let H(r) be EVEN, then the prime sum is -2 Σ Λ(n)/sqrt(n) g(log n) with g(u) = (1/2π) ∫ H(r) e^{-iru} dr.
# For even H, g(u) is real and g(-u) = g(u).
# 
# For NON-EVEN h, decompose h = h_even + h_odd. The EF sum Σ_γ h(γ) = Σ_γ h_even(γ) + Σ_γ h_odd(γ).
# Since zeros come in ±γ pairs (ζ self-dual), Σ_γ h_odd(γ) = 0 by symmetry.
# So Σ_γ h(γ) = Σ_γ h_even(γ) = (EF applied to h_even).
# h_even(r) = (h(r)+h(-r))/2.
# 
# So we should use h_even = ψ_even(r) = (ψ(r) + ψ(-r))/2. But ψ(-r) ≈ 0 since ψ concentrated at T0>0.
# To good approximation Σ_γ ψ(γ) = Σ_γ ψ_even(γ) and prime sum uses g_even = real part of our g (because 
# g_even(u) = (1/2π) ∫ ψ_even(r) e^{-iru} dr = (1/2π) ∫ ψ_even(r) cos(ru) dr = real).
# Or more directly: g_even(u) = (g(u) + g(-u))/2 = Re(g(u)) for real ψ.

# But for h not symmetric, the natural EF requires sum over all γ (positive AND negative).
# For ζ: Σ_{γ>0} h(γ) + Σ_{γ>0} h(-γ) = Σ_{γ>0} [h(γ) + h(-γ)] = 2 Σ_{γ>0} h_even(γ).
# So the EF applied to h_even: 2 Σ_{γ>0} h_even(γ) = arith side(h_even).
# Our M_zeros[i,j] = Σ_{γ>0} φ_i(γ) φ_j(γ) (assuming we sum over positive γ).
# Actually let me reconsider: M_zeros is conventionally over all zeros, both positive and negative γ.
# For our trace, tr(M_zeros) = Σ_{γ all} ψ(γ) ≈ Σ_{γ>0} ψ(γ) since ψ(γ<0) ≈ 0.
# 
# Equivalently: arith side as written sums over all γ.

# Let me just compute the EF for ζ.
# Arith side(ψ):
# + h(i/2) + h(-i/2) ≈ 0
# - g(0) log π = - (J/2π) log π
# + I_arch = (1/2π) ∫ψ(r) Re ψ_d(1/4 + ir/2) dr
# - 2 Σ Λ(n)/sqrt(n) g(log n) [this sum is complex in general; for real h on real line, sum = 2 Re Σ ...]

# For real h(r): Σ_γ h(γ) is real. So we need the real part of -2 Σ ...
# Actually since the EF is an EQUALITY, the RHS must be real if LHS is real. The -2 Σ part is in fact 
# 2 Re version because g(-u) = conj(g(u)) for real h, and the sum over n is summed only for n≥2, with implicit ±.
# Wait no — the standard EF uses ONLY n≥2 (positive logs). The other "side" came from the f.e. of L.
# So the formula is exactly: - 2 Σ_{n≥2} Λ(n)/sqrt(n) * Re(g(log n)) [actually you take 2*Re(Σ g(log n) Λ(n)/sqrt(n)) and stuff]

# To avoid confusion: derive via Mellin. The standard Weil form for ζ:
# Σ_γ h(γ) = h(i/2) + h(-i/2) - g(0) log π + (1/2π) ∫ h(r) Re ψ_d(1/4 + ir/2) dr - 2 Σ_n Λ(n)/sqrt(n) g(log n)
# where h is EVEN and g is its FT (real). For h non-even, the formula extends but the prime sum becomes:
# 2 Σ_n Λ(n)/sqrt(n) (1/2)[g(log n) + g(-log n)] = Σ_n Λ(n)/sqrt(n) [g(log n) + g(-log n)]
# = Σ_n Λ(n)/sqrt(n) * 2 Re(g(log n)) [since g(-u) = conj(g(u)) for real h]

# So prime sum contribution = -2 Σ Λ(n)/sqrt(n) Re(g(log n)) for real h.
# Wait the factor of 2 vs single? Let me be careful.
# For EVEN h: Σ_γ h(γ) = ... - 2 Σ_n Λ(n)/sqrt(n) g(log n)
# Here EF sums γ over both positive and negative. Both contribute same since h even.
# For real but not even: there are TWO formulas — one for each "half" of the spectrum.
# 
# Direct from Riemann-Weil derivation: 
# (1/2πi) ∫_{(c)} h_Mellin(s)... 
# 
# OK, fastest derivation: Σ_γ h(γ) = (poles) + integral_archimedean - Σ_n c_n g_h(log n) where c_n are 
# explicit and g_h = FT of h.
# For h NOT necessarily even, the precise form is:
# Σ_γ h(γ) = h(i/2) + h(-i/2) - g(0) log π + (1/2π) ∫h(r) Re ψ_d(1/4+ir/2) dr 
# - Σ_n Λ(n)/sqrt(n) [g(log n) + g(-log n)]
# (this becomes 2 Σ g if g even.)
# 
# For real h: g(-u) = conj(g(u)). So [g(log n) + g(-log n)] = 2 Re g(log n).
# Hence prime sum = - 2 Σ_n Λ(n)/sqrt(n) Re g(log n).

# Let me compute:
from sympy import isprime

# Primes up to some cutoff. Given Gaussian suppression in g(log p), can use X = 1000 (safely).
# Actually let's verify the cutoff needed for 1e-15 precision.
# Λ(n)/sqrt(n) Re g(log n) at n=1000: Re g(log 1000) ~ -3.7e-11 from our test above.
# Λ(1000)=0 (not prime power). At p=997 (prime), Λ/sqrt(p) ~ log(997)/31.6 ~ 0.218.
# Contribution: 0.218 * 3.7e-11 ~ 8e-12. So X=1000 gives ~8e-12 error.
# Need X=10^5 for 1e-15 (Gaussian falls super-exp).

# At p=10^5, log p = 11.5. |g(11.5)| ~ exp(-σ²(log p - 0)²/2)... wait g concentrates at u=0 with width 1/σ for Gaussian h.
# Actually g(u) = e^{-iT0 u} times function decaying as Gaussian of width 1/σ. So |g(u)| ~ exp(-σ²u²/2)*(stuff).
# At u=log(10^5)=11.5: |g| ~ exp(-2*132)/something = 1e-115. Negligible.
# 
# Let me compute prime sum up to X=10^5.
X = 100000
from sympy import sieve
primes_list = list(sieve.primerange(2, X+1))
print(f"Number of primes ≤ {X}: {len(primes_list)}")

Number of primes ≤ 100000: 9592


In [48]:
# Computing g(log p) for 9592 primes via quad is slow. Let me use analytic form.
# h(r) = Σ_j |φ_j(r)|² with φ_j(r) = (1/sqrt(σ)) h_j_hg((r-T0)/σ).
# Let x = (r-T0)/σ. Then φ_j(r) = h_j_hg(x)/sqrt(σ).
# φ_j² = h_j_hg(x)² / σ.
# 
# So ψ(r) = (1/σ) Σ_j h_j_hg((r-T0)/σ)².
# 
# Define K_J(x) = Σ_{j=0}^{J-1} h_j_hg(x)². This is the Christoffel-Darboux-type kernel.
# It has the form (Mehler kernel approach) but exact closed form:
# K_J(x) = something involving Hermite polynomials.
# 
# Fourier transform: g(u) = (1/2π) ∫ ψ(r) e^{-iru} dr = e^{-iT0 u}/(2π) * ∫ K_J(x)/σ * e^{-iσ x u} σ dx
# = e^{-iT0 u}/(2π) ∫ K_J(x) e^{-iσx u} dx 
# = e^{-iT0 u} K̂_J(σu) where K̂_J(k) = (1/2π) ∫ K_J(x) e^{-ikx} dx.

# h_j_hg(x) FT = (-i)^j h_j_hg(x) (Hermite-Gauss eigenfunctions of FT with factor (-i)^j when using physicists' Hermite normalized).
# h_j_hg(x)² FT: more complex. Let me skip analytic form and just compute numerically per prime, but vectorize.

# Numerical strategy: precompute g(log p) for primes. Use mpmath quad with caching.
# Time-critical: just iterate.

# Use lower precision dps=30 for the prime sum — 1e-15 target only.
mp.mp.dps = 30

# Vectorize g via direct mpmath integration. But 9592 calls × quad ≈ 9592 × 0.5s = 4800s. Too slow!
# 
# Use analytic Fourier transform of ψ for fast evaluation.
# ψ(r) = (1/σ) Σ_{j=0}^{J-1} h_j_hg((r-T0)/σ)²
# Let y = (r-T0)/σ, dr = σ dy.
# g(u) = (1/2π) ∫_{-∞}^∞ ψ(r) e^{-iru} dr
# = (1/2π) (1/σ) σ ∫ Σ h_j_hg(y)² e^{-i(σy+T0)u} dy
# = (e^{-iT0 u}/2π) Σ_j ∫ h_j_hg(y)² e^{-iσyu} dy
# 
# F_j(k) := ∫_{-∞}^∞ h_j_hg(y)² e^{-iky} dy.
# Since h_j_hg(y)² is real and even (because h_j is even or odd, so h_j² always even):
# F_j(k) = ∫ h_j_hg(y)² cos(ky) dy.
# 
# Formula (known): ∫ H_j(y)² exp(-y²) cos(ky) dy = sqrt(π) e^{-k²/4} L_j(k²/2) where L_j is Laguerre.
# Actually: ∫ H_j(y)² exp(-y²) dy = 2^j j! sqrt(π) (normalization).
# More general: ∫ H_m(y) H_n(y) exp(-y²) e^{iky} dy ... there's a formula.
# 
# Recall: h_j_hg(y) = (1/sqrt(2^j j! sqrt(π))) H_j(y) exp(-y²/2).
# h_j_hg(y)² = (1/(2^j j! sqrt(π))) H_j(y)² exp(-y²).
# So F_j(k) = (1/(2^j j! sqrt(π))) ∫ H_j(y)² exp(-y²) e^{-iky} dy.
# 
# Known integral: ∫ H_m(y) H_n(y) e^{-y²} e^{-iky} dy = ?
# For m=n: ∫ H_n(y)² e^{-y²} e^{-iky} dy = sqrt(π) 2^n n! e^{-k²/4} L_n(k²/2)
# (where L_n is Laguerre polynomial). 
# 
# So F_j(k) = (1/(2^j j! sqrt(π))) * sqrt(π) 2^j j! e^{-k²/4} L_j(k²/2) = e^{-k²/4} L_j(k²/2).
# 
# So g(u) = (e^{-iT0 u}/2π) Σ_j e^{-(σu)²/4} L_j((σu)²/2)
# = (e^{-iT0 u} e^{-σ²u²/4} / 2π) Σ_{j=0}^{J-1} L_j(σ²u²/2)
# Sum of Laguerre polynomials: Σ_{j=0}^{J-1} L_j(x) = L_{J-1}^{(1)}(x) (associated Laguerre).
# Actually identity: Σ_{j=0}^{N} L_j(x) = L_N^{(1)}(x).
# So Σ_{j=0}^{J-1} L_j(x) = L_{J-1}^{(1)}(x).
# 
# Hence g(u) = (e^{-iT0 u} e^{-σ²u²/4} / 2π) L_{J-1}^{(1)}(σ²u²/2)

# Verify
from sympy.functions.special.polynomials import assoc_laguerre, laguerre
def g_analytic(u):
 u_mp = mp.mpf(u) if not isinstance(u, mp.mpf) else u
 sigma_u_sq = (sigma * u_mp)**2
 x = sigma_u_sq / 2
 # L_{J-1}^{(1)}(x) — associated Laguerre
 # Use mpmath.laguerre with α parameter
 L = mp.laguerre(J-1, 1, x)
 return mp.exp(-mp.mpc(0, T0*u_mp)) * mp.exp(-sigma_u_sq/4) / (2*mp.pi) * L

mp.mp.dps = 50
# Compare with numerical
for u_val in [0, 0.5, 1, mp.log(2), mp.log(10)]:
 g_num = g_func(mp.mpf(u_val))
 g_ana = g_analytic(mp.mpf(u_val))
 print(f"u={float(u_val):.3f}: g_num={complex(g_num):.6e}, g_ana={complex(g_ana):.6e}")

u=0.000: g_num=1.591549e+00+0.000000e+00j, g_ana=1.591549e+00+0.000000e+00j


u=0.500: g_num=-6.688355e-02-1.426245e-01j, g_ana=-6.688355e-02-1.426245e-01j


u=1.000: g_num=-5.937635e-02+7.138800e-02j, g_ana=-5.937635e-02+7.138800e-02j


u=0.693: g_num=1.180930e-01+3.492521e-02j, g_ana=1.180930e-01+3.492521e-02j


u=2.303: g_num=-7.416435e-04-4.952780e-04j, g_ana=-7.416435e-04-4.952780e-04j


In [49]:
# Analytic formula matches numerical perfectly. Now compute prime sum efficiently.
mp.mp.dps = 50

# Prime sum: -2 Σ_n Λ(n)/sqrt(n) Re g(log n) for ζ.
# n = p^k for primes p, integer k ≥ 1. Λ(n) = log p.
# Only n ≤ X = 10^5.

t0 = time.time()
prime_sum_zeta = mp.mpf(0)
for p in primes_list:
 log_p = mp.log(p)
 # k=1, 2, ... while p^k ≤ X
 pk = p
 k = 1
 while pk <= X:
 u = k * log_p
 g_val = g_analytic(u)
 prime_sum_zeta += log_p / mp.sqrt(pk) * mp.re(g_val)
 pk *= p
 k += 1

prime_sum_zeta_contrib = -2 * prime_sum_zeta
print(f"Prime sum contribution (ζ) = {prime_sum_zeta_contrib}")
print(f"Time: {time.time()-t0:.2f}s")

Prime sum contribution (ζ) = -0.25820322477302995575996305325701324759583411137434
Time: 1.20s


In [50]:
# Now compute polar term and archimedean for ζ
# (ζ has poles via complete Lambda_ζ(s) = π^{-s/2} Γ(s/2) ζ(s); pole of ζ at s=1, with completed Λ having simple poles 
# at s=0 and s=1.)
# Polar contribution: h(i/2) + h(-i/2). Where h = ψ, evaluated at s=0,1 corresponding to imaginary part γ=i/2, -i/2.
# In our convention zero is ρ = 1/2 + iγ. Pole of ζ at s=1 ↔ ρ=1 ↔ iγ = 1/2 ↔ γ = -i/2.
# Pole at s=0 ↔ ρ=0 ↔ iγ = -1/2 ↔ γ = i/2.
# So pole contribution = h(i/2) + h(-i/2) = ψ(i/2) + ψ(-i/2).

# Evaluate ψ at complex t = ±i/2.
# Use the analytic continuation: phi_n_t(n, t) defined formally, hermite_phys returns complex.

def psi_complex(t_c):
 s = mp.mpc(0)
 for j in range(J):
 s += phi_n_t(j, t_c)**2
 return s

mp.mp.dps = 50
psi_i_half = psi_complex(mp.mpc(0, mp.mpf("0.5")))
psi_neg_i_half = psi_complex(mp.mpc(0, mp.mpf("-0.5")))
print(f"ψ(i/2) = {psi_i_half}")
print(f"ψ(-i/2) = {psi_neg_i_half}")
print(f"Sum (polar) = {psi_i_half + psi_neg_i_half}")
print(f"|Polar| = {abs(psi_i_half + psi_neg_i_half)}")

ψ(i/2) = (-2.9560280324083163404420903425981701653569800188625e-772 + 2.3485035848018741702380871786222445744955132300296e-772j)
ψ(-i/2) = (-2.9560280324083163404420903425981701653569800188625e-772 - 2.3485035848018741702380871786222445744955132300296e-772j)
Sum (polar) = (-5.9120560648166326808841806851963403307139600377251e-772 + 0.0j)
|Polar| = 5.9120560648166326808841806851963403307139600377251e-772


In [51]:
# Polar terms are negligible (10^{-772}). Good.

# Now assemble arithmetic side for ζ:
# tr(M_arith) for ζ = Σ_γ ψ(γ)_arith side
# = polar + (-g(0) log π) + I_arch + prime_sum_contrib
# But wait — there's an additional negative-γ contribution from the (small) tail of zeros at γ < 0.
# Since these are negligibly small for our ψ centered at T0=85.7, they contribute ~0.

g0 = mp.mpf(J) / (2*mp.pi) # = g(0)
arith_polar_zeta = psi_i_half + psi_neg_i_half # complex, tiny
arith_g0_logpi = -g0 * mp.log(mp.pi)
arith_zeta = arith_polar_zeta + arith_g0_logpi + I_arch + prime_sum_zeta_contrib
print(f"Arithmetic side for ζ:")
print(f" Polar: {arith_polar_zeta}")
print(f" -g(0) log π: {arith_g0_logpi}")
print(f" Archimedean integral: {I_arch}")
print(f" Prime sum (-2 Σ): {prime_sum_zeta_contrib}")
print(f" Total tr(M_arith) = {arith_zeta}")
print(f"")
print(f"tr(M_zeros) for ζ = {tr_zeros_zeta}")
residual = abs(tr_zeros_zeta - arith_zeta.real if isinstance(arith_zeta, mp.mpc) else tr_zeros_zeta - arith_zeta)
print(f"|tr(M_zeros) - tr(M_arith)| = {residual}")

Arithmetic side for ζ:
 Polar: (-5.9120560648166326808841806851963403307139600377251e-772 + 0.0j)
 -g(0) log π: -1.821894198379531285247938651580812069458535195493
 Archimedean integral: 5.9783922212210922281828970530543552361164737571726
 Prime sum (-2 Σ): -0.25820322477302995575996305325701324759583411137434
 Total tr(M_arith) = (3.8982947980685309871749953482165299190621044503052 + 0.0j)

tr(M_zeros) for ζ = 3.8982947980685309871749953482165299190621046029871
|tr(M_zeros) - tr(M_arith)| = 1.5268184442989928181751691411821643141623085068781e-43


In [52]:
# EXCELLENT! Trace identity for ζ holds to ~10^{-43} precision — far exceeding the 10^{-15} target!
# GATE PASSED for ζ.

# Now do the same for L(χ mod 5).
# Archimedean for L(χ), χ odd primitive mod 5, a=1:
# γ_∞(s) = (π/5)^{-(s+1)/2} Γ((s+1)/2) -- wait we should include level q.
# Λ(s, χ) = (q/π)^{(s+1)/2} Γ((s+1)/2) L(s, χ).
# Equivalent Γ-factor (with level): (q/π)^{s/2} (π)^{-(a)/2} Γ((s+a)/2) — convention varies. Let me derive 
# the EF in our normalization.
# 
# log derivative of completed Λ_χ(s) = (q/π)^{(s+a)/2} Γ((s+a)/2) L(s,χ):
# Λ'/Λ (s) = (1/2) log(q/π) + (1/2) ψ_d((s+a)/2) + L'/L (s, χ)
# 
# Hadamard product: Λ(s) = e^{A+Bs} Π_ρ (1 - s/ρ) e^{s/ρ}
# Λ'/Λ (s) = B + Σ_ρ (1/(s-ρ) + 1/ρ)
# 
# Explicit formula (analog of Riemann-Weil): for test function h(r) with FT g(u):
# Σ_γ h(γ) = - g(0) log(q/π) [from the constant in log derivative, with factor 2]
# + (1/2π) ∫ h(r) Re ψ_d((1/2 + a + ir)/2) dr [from Γ-factor]
# - 2 Σ_n Λ_χ(n)/sqrt(n) g(log n) [from L'/L, with sign]
# where Λ_χ(n) = χ(p^k) log p for n = p^k.
# 
# For L(χ) PRIMITIVE there are NO polar terms (L(s,χ) is entire).
# 
# Note: -g(0) log(q/π) — for q=5: log(5/π) ≈ log(1.59) ≈ 0.464.
# 
# Wait. Let me re-derive carefully. For ζ we had:
# (constant term from Γ_∞): The constant log π was negative because π comes in the denominator of (1/π)^{s/2}.
# Specifically Λ_ζ(s) = π^{-s/2} Γ(s/2) ζ(s). The (1/π)^{s/2} contributes -g(0) (1/2) log π * 2 = -g(0) log π... 
# Wait my formula gives -g(0) log π which was applied as one term. Hmm.
# 
# For L(χ): Λ_χ(s) = (q/π)^{(s+a)/2} Γ((s+a)/2) L(s,χ). 
# log Λ' = (1/2) log(q/π) + (1/2) ψ_d((s+a)/2) + L'/L
# When deriving EF, this constant gets paired with the (1/2π) ∫ contribution, and integrated over h.
# Using residue / contour integration: 
# Σ_γ h(γ) = (1/2π) ∫ h(r) [log(q/π) + Re ψ_d((1/2+a+ir)/2)] dr - 2 Σ Λ_χ(n)/sqrt(n) Re g(log n)
# [for h real]
# So the constant log(q/π) term is now POSITIVE (no minus sign, since the constant comes from the OUTSIDE 
# part of the contour).
# 
# Wait this contradicts my ζ formula. Let me re-derive ζ:
# Λ_ζ(s) = π^{-s/2} Γ(s/2) ζ(s) * (s(s-1)/2)^{-1}? No, just π^{-s/2} Γ(s/2) ζ(s).
# log derivative: -log(π)/2 + ψ_d(s/2)/2 + ζ'/ζ
# 
# For ζ: there are poles at s=0,1 of Λ giving pole terms. Plus the integrand contribution.
# Σ_γ h(γ) = h(γ at pole) + (1/2π) ∫ h(r) [-log π + Re ψ_d(1/4 + ir/2)] dr - 2 Σ Λ(n)/sqrt(n) g(log n)
# = polar - g(0) log π + I_arch - 2 prime sum.
# OK so my formula for ζ has -g(0) log π. Sign is NEGATIVE.
# 
# For L(χ): Λ_χ(s) constant log derivative is +(1/2) log(q/π). So contribution: +g(0) log(q/π).
# Sign opposite to ζ because q/π > 0 vs 1/π for ζ. 
# 
# Actually for ζ we had (1/π)^{s/2}, which has log-derivative -log(π)/2. So in EF: -g(0) * 2 * (-log π /2) = +g(0) log π?
# No I'm confusing myself. Let me just look at the actual signs.
# 
# In ζ result: arith side = -g(0) log π + I_arch + prime + polar
# Numerical: -1.8219 + 5.978 - 0.258 + ~0 = 3.898 ✓ matches zero side 3.898.
# So formula correct.

# For L(χ): Λ(s) = (q/π)^{(s+a)/2} Γ((s+a)/2) L(s,χ). NO poles (L primitive).
# log derivative: (1/2) log(q/π) + (1/2) ψ_d((s+a)/2) + L'/L.
# In Weil EF, the structure becomes:
# Σ_γ h(γ) = + g(0) log(q/π) + (1/2π) ∫ h(r) Re ψ_d((1/2+a+ir)/2) dr - 2 Σ_n Λ_χ(n)/sqrt(n) Re g(log n)
# 
# Wait, for ζ: Λ_ζ has the constant (1/π)^{s/2}, which has log-deriv = -log(π)/2. The Σ_γ h(γ) formula gives 
# constant term = -g(0) log π. So the EF sign is +2*g(0) * (log-deriv constant). I.e., +g(0)*(-log π) = -g(0) log π.
# 
# So for L(χ): +g(0) * log(q/π) where (1/2) log(q/π) is the constant in log-deriv (factor 2 in EF gives g(0) log(q/π)).
# Sign matches if I view: log Λ' = (1/2) log(q/π) + ..., then contribution to EF = 2 * (1/2) log(q/π) * g(0) = log(q/π) * g(0).
# 
# But this contradicts my ζ derivation where I said -g(0) log π. Let me see: ζ has constant (1/π)^{s/2} → log-deriv constant -log(π)/2 → EF contribution 2 * (-log π /2) * g(0) = -g(0) log π. OK that's consistent.
# For L(χ) mod 5 (odd, a=1, q=5): constant (5/π)^{(s+1)/2} → log-deriv constant (1/2) log(5/π) → EF contribution g(0) log(5/π) = g(0) [log 5 - log π].

# So:
g0_logqpi = g0 * mp.log(mp.mpf(5)/mp.pi) # = +g(0) log(5/π) for L(χ mod 5)
# Archimedean integral
def arch_integrand_Lchi(r):
 r_mp = mp.mpf(r) if not isinstance(r, mp.mpf) else r
 # Re ψ_d((1/2 + a + ir)/2) = Re ψ_d((1/2 + 1 + ir)/2) = Re ψ_d(3/4 + ir/2)
 return psi_func(r_mp) * mp.re(mp.digamma(mp.mpf("0.75") + mp.mpc(0, r_mp/2)))

I_arch_Lchi = mp.quad(arch_integrand_Lchi, [t_lo, t_hi]) / (2*mp.pi)
print(f"I_arch (L(χ)) = {I_arch_Lchi}")
print(f"g(0) log(5/π) = {g0_logqpi}")

I_arch (L(χ)) = 5.9783922212210922281828970530543552361164737571726
g(0) log(5/π) = 0.73960579525434945213371632652547285838548142117514


In [53]:
# Hmm I_arch(Lchi) = I_arch(zeta)? That seems wrong. Let me check digamma values for L(chi) (which uses 
# 3/4 + ir/2) vs zeta (1/4 + ir/2).
print(f"Re ψ_d(3/4 + 50i) = {mp.re(mp.digamma(mp.mpf('0.75') + mp.mpc(0, 50)))}")
print(f"Re ψ_d(1/4 + 50i) = {mp.re(mp.digamma(mp.mpf('0.25') + mp.mpc(0, 50)))}")
# For large |Im|, ψ_d(σ + it) ~ log(t) + 1/(2t)... approximately equal for σ=1/4 vs 3/4.
# So I_arch_Lchi ≈ I_arch_zeta makes sense — they only differ in low-order terms.

Re ψ_d(3/4 + 50i) = 3.9120188386885588806258986195095630969175972936462
Re ψ_d(1/4 + 50i) = 3.9120188386885588806258986195095630969175972936462
